In [ ]:
CONFIG = {
    "dataset": 'NFCorpus',
    "dataset_url": "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip",
    "results_dir": "results",
    "dense_model": "sentence-transformers/all-MiniLM-L6-v2",
    "dense_batch_size": 64,
    "top_k": 10,
    "top_k_eval": [1, 3, 5, 10],
    "delta": 0.05,
    "n_bootstrap": 10000
}
SEED = 5


COLORS = {
    "BM25": "#e74c3c",
    "Dense": "#3498db",
    "Hybrid": "#2ecc71",
}

# Colours for additional dense baselines
COLORS["BGE-small"] = "#9b59b6"
COLORS["E5-small"]  = "#f39c12"


In [ ]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import os


data_path = os.path.join("datasets", CONFIG["dataset"])
if not os.path.exists(data_path):
    print("Downloading NFCorpus...")
    data_path = util.download_and_unzip(CONFIG["dataset_url"], "datasets")
    print(f"Downloaded to: {data_path}")
else:
    print(f"Dataset already exists at: {data_path}")


corpus, test_queries, test_qrels = GenericDataLoader(data_path).load(split="test")
_, train_queries, train_qrels = GenericDataLoader(data_path).load(split="train")
_, dev_queries, dev_qrels = GenericDataLoader(data_path).load(split="dev")

print(f"\n{'='*50}")
print(f"  NFCorpus Dataset Summary")
print(f"{'='*50}")

print(f"  Corpus:        {len(corpus):,} documents")
print(f"  Train queries: {len(train_queries):,}  |  Train qrels: {sum(len(v) for v in train_qrels.values()):,}")
print(f"  Dev queries:   {len(dev_queries):,}  |  Dev qrels:   {sum(len(v) for v in dev_qrels.values()):,}")
print(f"  Test queries:  {len(test_queries):,}  |  Test qrels:  {sum(len(v) for v in test_qrels.values()):,}")
print(f"{'='*50}")

In [ ]:
doc_ids = list(corpus.keys())
doc_texts = [
    (corpus[did].get("title", "") + " " + corpus[did].get("text", "")).strip()
    for did in doc_ids
]

print(f"Prepared {len(doc_ids):,} documents for indexing.")
print(f"\nExample document (first 300 chars):")
print(f"  ID:   {doc_ids[0]}")
print(f"  Text: {doc_texts[0][:300]}...")

In [ ]:
from collections import Counter
import numpy as np


doc_lengths = [len(text.split()) for text in doc_texts]

query_lengths = [len(q.split()) for q in test_queries.values()]

all_relevances = []
for qid in test_qrels:
    for did, rel in test_qrels[qid].items():
        all_relevances.append(rel)
rel_counts = Counter(all_relevances)

rels_per_query = [len(test_qrels[qid]) for qid in test_qrels]

print(f"Document Length Statistics:")
print(f"  Mean:   {np.mean(doc_lengths):.1f} words")
print(f"  Median: {np.median(doc_lengths):.1f} words")
print(f"  Min:    {np.min(doc_lengths)} words")
print(f"  Max:    {np.max(doc_lengths)} words")

print(f"\nQuery Length Statistics:")
print(f"  Mean:   {np.mean(query_lengths):.1f} words")
print(f"  Median: {np.median(query_lengths):.1f} words")

print(f"\nRelevance Distribution (test split):")
for rel_level in sorted(rel_counts.keys()):
    count = rel_counts[rel_level]
    print(f"  Level {rel_level}: {count:,} ({count/len(all_relevances)*100:.1f}%)")

print(f"\nRelevant Docs per Query:")
print(f"  Mean:   {np.mean(rels_per_query):.1f}")
print(f"  Median: {np.median(rels_per_query):.1f}")
print(f"  Max:    {np.max(rels_per_query)}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Plot 1: Document length distribution
axes[0].hist(doc_lengths, bins=50, color="#34495e", alpha=0.7, edgecolor="white")
axes[0].axvline(np.median(doc_lengths), color="red", linestyle="--",
                label=f"Median: {np.median(doc_lengths):.0f}")
axes[0].set_xlabel("Document Length (words)")
axes[0].set_ylabel("Count")
axes[0].set_title("Document Length Distribution")
axes[0].legend()

# Plot 2: Query length distribution
axes[1].hist(query_lengths, bins=20, color="#2980b9", alpha=0.7, edgecolor="white")
axes[1].axvline(np.median(query_lengths), color="red", linestyle="--",
                label=f"Median: {np.median(query_lengths):.0f}")
axes[1].set_xlabel("Query Length (words)")
axes[1].set_ylabel("Count")
axes[1].set_title("Query Length Distribution")
axes[1].legend()

# Plot 3: Relevance distribution
rel_labels = [f"Level {k}" for k in sorted(rel_counts.keys())]
rel_values = [rel_counts[k] for k in sorted(rel_counts.keys())]
bars = axes[2].bar(rel_labels, rel_values, color=["#e74c3c", "#f39c12", "#27ae60"],
                   edgecolor="white")
for bar, val in zip(bars, rel_values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f"{val:,}", ha="center", fontsize=10)
axes[2].set_xlabel("Relevance Level")
axes[2].set_ylabel("Count")
axes[2].set_title("Relevance Distribution (Test)")

plt.tight_layout()
os.makedirs(CONFIG["results_dir"], exist_ok=True)
plt.savefig(os.path.join(CONFIG["results_dir"], "dataset_exploration.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

In [ ]:
print("=" * 70)
print("  Example Queries with Relevant Documents")
print("=" * 70)

example_count = 0
for qid, query_text in test_queries.items():
    if qid not in test_qrels or example_count >= 3:
        break

    print(f"\n  Query [{qid}]: \"{query_text}\"")
    print(f"  Number of relevant documents: {len(test_qrels[qid])}")

    # Show top 2 relevant docs sorted by relevance
    sorted_rels = sorted(test_qrels[qid].items(), key=lambda x: -x[1])
    for did, rel in sorted_rels[:2]:
        title = corpus[did].get("title", "No title")
        text_preview = corpus[did].get("text", "")[:120]
        print(f"    → [{did}] relevance={rel}")
        print(f"      Title: {title}")
        print(f"      Text:  {text_preview}...")

    example_count += 1

print("\n" + "=" * 70)


---
## 2. Building Retrieval Systems

We build three retrievers, each with a fundamentally different inductive bias:

| Retriever | Inductive Bias | Hypothesis |
|-----------|---------------|------------|
| BM25 | Exact lexical matching | "If the query words appear in the document, it is relevant" |
| Dense | Semantic embedding similarity | "If texts mean similar things, they are relevant" |
| Hybrid | Combined lexical + semantic | "Relevance requires both word overlap AND meaning similarity" |

Each retriever is a function h: Q → ranked list of documents.


In [ ]:
import nltk
from rank_bm25 import BM25Okapi
import time
from typing import List, Tuple, Dict


try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

print("Building BM25 retriever...")
print("  Inductive bias: exact lexical matching")
print("  Assumption: relevance = term frequency × inverse document frequency")
print()

# ── Tokenize corpus ──────────────────────────────────────────
t0 = time.time()

tokenized_corpus = [nltk.word_tokenize(doc.lower()) for doc in doc_texts]

bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

bm25_index_time = time.time() - t0
print(f"  Index built in {bm25_index_time:.2f}s")
print(f"  Vocabulary coverage: bag-of-words over {len(doc_ids):,} documents")
print(f"  Parameters: k1=1.5, b=0.75")


# ── Retrieval function ───────────────────────────────────────
def bm25_retrieve(query: str, top_k: int = 10) -> List[Tuple[str, float]]:
    """Retrieve top-k documents using BM25 scoring."""
    tokenized_query = nltk.word_tokenize(query.lower())
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return [(doc_ids[idx], float(scores[idx])) for idx in top_indices]


def bm25_retrieve_batch(
    queries: Dict[str, str], top_k: int = 10
) -> Dict[str, Dict[str, float]]:
    """Batch retrieval for all queries. Returns BEIR-compatible format."""
    results = {}
    for qid, query_text in queries.items():
        retrieved = bm25_retrieve(query_text, top_k)
        results[qid] = {doc_id: score for doc_id, score in retrieved}
    return results


# ── Quick test ───────────────────────────────────────────────
test_query = list(test_queries.values())[0]
test_results = bm25_retrieve(test_query, top_k=3)
print(f"\n  Test query: \"{test_query}\"")
print(f"  Top 3 results:")
for rank, (did, score) in enumerate(test_results):
    title = corpus[did].get("title", "N/A")[:60]
    print(f"    {rank+1}. [{did}] score={score:.3f} | {title}")

print("\n✓ BM25 retriever ready.")

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

print("Building Dense retriever...")
print("  Model: all-MiniLM-L6-v2 (22M parameters, 384 dimensions)")
print("  Inductive bias: semantic similarity in embedding space")
print("  Assumption: cosine similarity of embeddings ≈ relevance")
print()

# ── Load encoder ─────────────────────────────────────────────
dense_model = SentenceTransformer(CONFIG["dense_model"])
print(f"  Encoder loaded: {CONFIG['dense_model']}")

# ── Encode corpus ────────────────────────────────────────────
t0 = time.time()

doc_embeddings = dense_model.encode(
    doc_texts,
    batch_size=CONFIG["dense_batch_size"],
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

dense_encode_time = time.time() - t0
print(f"\n  Corpus encoded in {dense_encode_time:.2f}s")
print(f"  Embedding matrix shape: {doc_embeddings.shape}")

embedding_dim = doc_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(doc_embeddings)
print(f"  FAISS index built: {faiss_index.ntotal} vectors, dim={embedding_dim}")


# ── Retrieval functions ──────────────────────────────────────
def dense_retrieve(query: str, top_k: int = 10) -> List[Tuple[str, float]]:
    """Retrieve top-k documents by cosine similarity."""
    q_emb = dense_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    scores, indices = faiss_index.search(q_emb, top_k)
    return [
        (doc_ids[indices[0][i]], float(scores[0][i]))
        for i in range(top_k)
        if indices[0][i] < len(doc_ids)
    ]


def dense_retrieve_batch(
    queries: Dict[str, str], top_k: int = 10
) -> Dict[str, Dict[str, float]]:
    """Efficient batch retrieval using FAISS batch search."""
    qid_list = list(queries.keys())
    query_texts = [queries[qid] for qid in qid_list]

    q_embeddings = dense_model.encode(
        query_texts,
        batch_size=CONFIG["dense_batch_size"],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, indices = faiss_index.search(q_embeddings, top_k)

    results = {}
    for i, qid in enumerate(qid_list):
        results[qid] = {}
        for j in range(top_k):
            idx = indices[i][j]
            if 0 <= idx < len(doc_ids):
                results[qid][doc_ids[idx]] = float(scores[i][j])
    return results

In [ ]:
# ============================================================================
# P3: Additional dense baselines — BGE-small-en-v1.5 & E5-small-v2
#
# BGE (BAAI/bge-small-en-v1.5): strong MTEB-class bi-encoder, no prefix needed.
# E5  (intfloat/e5-small-v2)   : uses 'query:'/'passage:' prefixes.
# Both are free on HuggingFace and significantly outperform all-MiniLM-L6-v2.
# ============================================================================
import sys as _sys

try:
    import psutil as _psutil
except ImportError:
    import subprocess
    subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "psutil"], check=True)
    import psutil as _psutil


def build_dense_retriever(
    model_name: str,
    doc_texts: list,
    doc_ids: list,
    batch_size: int = 64,
    query_prefix: str = "",
    doc_prefix: str = "",
) -> dict:
    """
    Build a dense retriever for any SentenceTransformer model.
    Returns a dict with keys: model_name, model, index, embeddings,
    encode_time, index_memory_mb, retrieve (single), retrieve_batch.
    """
    import os as _os
    proc = _psutil.Process(_os.getpid())
    mem_before = proc.memory_info().rss / 1024 ** 2

    model = SentenceTransformer(model_name)

    t0 = time.time()
    corpus_texts = [doc_prefix + t for t in doc_texts] if doc_prefix else doc_texts
    embeddings = model.encode(
        corpus_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")
    encode_time = time.time() - t0

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    mem_after = proc.memory_info().rss / 1024 ** 2
    index_memory_mb = mem_after - mem_before

    # Capture state in closures
    _model, _index, _doc_ids, _qpfx = model, index, doc_ids, query_prefix
    _bs = batch_size

    def _retrieve(query: str, top_k: int = 10):
        q_text = (_qpfx + query) if _qpfx else query
        q_emb = _model.encode(
            [q_text], normalize_embeddings=True, convert_to_numpy=True
        ).astype("float32")
        scores, indices = _index.search(q_emb, top_k)
        return [
            (_doc_ids[indices[0][i]], float(scores[0][i]))
            for i in range(top_k)
            if indices[0][i] < len(_doc_ids)
        ]

    def _retrieve_batch(queries: dict, top_k: int = 10) -> dict:
        qid_list = list(queries.keys())
        qtexts = [
            ((_qpfx + queries[qid]) if _qpfx else queries[qid])
            for qid in qid_list
        ]
        q_embs = _model.encode(
            qtexts,
            batch_size=_bs,
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype("float32")
        scores_arr, indices_arr = _index.search(q_embs, top_k)
        results = {}
        for i, qid in enumerate(qid_list):
            results[qid] = {}
            for j in range(top_k):
                idx = indices_arr[i][j]
                if 0 <= idx < len(_doc_ids):
                    results[qid][_doc_ids[idx]] = float(scores_arr[i][j])
        return results

    return dict(
        model_name=model_name,
        model=model,
        index=index,
        embeddings=embeddings,
        encode_time=encode_time,
        index_memory_mb=index_memory_mb,
        retrieve=_retrieve,
        retrieve_batch=_retrieve_batch,
    )


print("Building BGE-small-en-v1.5 retriever...")
bge_retriever = build_dense_retriever(
    model_name="BAAI/bge-small-en-v1.5",
    doc_texts=doc_texts,
    doc_ids=doc_ids,
    batch_size=CONFIG["dense_batch_size"],
)
print(f"  Built in {bge_retriever['encode_time']:.1f}s  "
      f"| memory delta ≈ {bge_retriever['index_memory_mb']:.0f} MB")
print()

print("Building E5-small-v2 retriever (with 'query:'/'passage:' prefixes)...")
e5_retriever = build_dense_retriever(
    model_name="intfloat/e5-small-v2",
    doc_texts=doc_texts,
    doc_ids=doc_ids,
    batch_size=CONFIG["dense_batch_size"],
    query_prefix="query: ",
    doc_prefix="passage: ",
)
print(f"  Built in {e5_retriever['encode_time']:.1f}s  "
      f"| memory delta ≈ {e5_retriever['index_memory_mb']:.0f} MB")
print()
print("✓ Additional dense retrievers ready.")



---
## 3. Retrieval Evaluation — Standard IR Metrics

Before any generalization analysis, we compute standard information retrieval 
metrics to establish baseline performance:

- **nDCG@k**: Normalized Discounted Cumulative Gain — handles graded relevance
- **MAP@k**: Mean Average Precision — precision-oriented
- **Recall@k**: What fraction of relevant documents did we find?
- **Precision@k**: What fraction of retrieved documents are relevant?

We also compute the **per-query loss** ℓ(h, q) = 1 - nDCG@10, which is the
foundation for all generalization bound computations.


In [ ]:
from typing import List, Dict
import numpy as np

def compute_ndcg(ranked_docs: List[str], qrel: Dict[str, int], k: int) -> float:
    dcg = 0.0
    for i, doc_id in enumerate(ranked_docs[:k]):
        rel = qrel.get(doc_id, 0)
        dcg += (2 ** rel - 1) / np.log2(i + 2)

    # Ideal DCG
    ideal_rels = sorted(qrel.values(), reverse=True)[:k]
    idcg = sum((2 ** r - 1) / np.log2(i + 2) for i, r in enumerate(ideal_rels))

    return dcg / idcg if idcg > 0 else 0.0


def compute_ap(ranked_docs: List[str], qrel: Dict[str, int], k: int,
               threshold: int = 1) -> float:
    """Average Precision at k (binary: relevant if rel >= threshold)."""
    relevant = {d for d, r in qrel.items() if r >= threshold}
    if not relevant:
        return 0.0

    hits = 0
    ap = 0.0
    for i, doc_id in enumerate(ranked_docs[:k]):
        if doc_id in relevant:
            hits += 1
            ap += hits / (i + 1)

    return ap / min(len(relevant), k)


def compute_recall(ranked_docs: List[str], qrel: Dict[str, int], k: int,
                   threshold: int = 1) -> float:
    """Recall at k."""
    relevant = {d for d, r in qrel.items() if r >= threshold}
    if not relevant:
        return 0.0
    found = sum(1 for d in ranked_docs[:k] if d in relevant)
    return found / len(relevant)


def compute_precision(ranked_docs: List[str], qrel: Dict[str, int], k: int,
                      threshold: int = 1) -> float:
    """Precision at k."""
    relevant = {d for d, r in qrel.items() if r >= threshold}
    if k == 0:
        return 0.0
    found = sum(1 for d in ranked_docs[:k] if d in relevant)
    return found / k


def evaluate_retriever(
    results: Dict[str, Dict[str, float]],
    qrels: Dict[str, Dict[str, int]],
    k_values: List[int] = None,
) -> dict:
    """
    Complete evaluation of a retriever.
    
    Returns:
        - aggregate: averaged metrics across all queries
        - per_query: metrics for each individual query
        - loss_array: numpy array of per-query losses (1 - nDCG@10)
    """
    if k_values is None:
        k_values = CONFIG["top_k_eval"]

    per_query = {}
    per_query_losses = {}
    eval_qids = [qid for qid in qrels if qid in results]

    for qid in eval_qids:
        # Sort retrieved docs by score (descending)
        ranking = [
            doc_id for doc_id, score
            in sorted(results[qid].items(), key=lambda x: -x[1])
        ]
        qrel = qrels[qid]

        metrics = {}
        for k in k_values:
            metrics[f"nDCG@{k}"] = compute_ndcg(ranking, qrel, k)
            metrics[f"MAP@{k}"] = compute_ap(ranking, qrel, k)
            metrics[f"Recall@{k}"] = compute_recall(ranking, qrel, k)
            metrics[f"P@{k}"] = compute_precision(ranking, qrel, k)

        per_query[qid] = metrics
        per_query_losses[qid] = 1.0 - metrics["nDCG@10"]

    # Aggregate
    aggregate = {}
    if eval_qids:
        for metric_name in per_query[eval_qids[0]]:
            values = [per_query[qid][metric_name] for qid in eval_qids]
            aggregate[metric_name] = float(np.mean(values))

    loss_array = np.array([per_query_losses[qid] for qid in eval_qids])

    aggregate["empirical_risk"] = float(np.mean(loss_array))
    aggregate["loss_variance"] = float(np.var(loss_array, ddof=1))
    aggregate["loss_std"] = float(np.std(loss_array, ddof=1))
    aggregate["n_queries"] = len(eval_qids)

    return {
        "aggregate": aggregate,
        "per_query": per_query,
        "per_query_losses": per_query_losses,
        "loss_array": loss_array,
    }


print("✓ Evaluation functions defined.")

In [ ]:
from tqdm.auto import tqdm


TOP_K_RETRIEVE = 100

# ── BM25 ─────────────────────────────────────────────────────
print("  BM25...", end=" ")
t0 = time.time()
bm25_results = bm25_retrieve_batch(test_queries, top_k=TOP_K_RETRIEVE)
t_bm25 = time.time() - t0
print(f"done in {t_bm25:.2f}s  ({len(bm25_results)} queries)")

# ── Dense (with tqdm) ───────────────────────────────────────

print("  Dense...", end=" ")
t0 = time.time()
dense_results = {}
for qid, qtext in tqdm(test_queries.items(), desc="Dense retrieval", total=len(test_queries)):
    retrieved = dense_retrieve(qtext, top_k=TOP_K_RETRIEVE)
    dense_results[qid] = {doc_id: score for doc_id, score in retrieved}
t_dense = time.time() - t0
print(f"done in {t_dense:.2f}s  ({len(dense_results)} queries)")


print("\n✓ All retrieval complete.")

# ── BGE-small ─────────────────────────────────────────────────────────────
print("  BGE-small...", end=" ")
t0 = time.time()
bge_results = bge_retriever["retrieve_batch"](test_queries, top_k=TOP_K_RETRIEVE)
t_bge = time.time() - t0
print(f"done in {t_bge:.2f}s  ({len(bge_results)} queries)")

# ── E5-small ──────────────────────────────────────────────────────────────
print("  E5-small...", end=" ")
t0 = time.time()
e5_results = e5_retriever["retrieve_batch"](test_queries, top_k=TOP_K_RETRIEVE)
t_e5 = time.time() - t0
print(f"done in {t_e5:.2f}s  ({len(e5_results)} queries)")

print("\n✓ All retrieval complete.")


In [ ]:
# ============================================================================
# P4: Practical Efficiency Metrics
# Modern IR papers are judged on quality–efficiency trade-offs.
# We report: index build time, per-query latency, throughput, memory footprint.
# ============================================================================
import pandas as _pd


def _throughput(retrieve_batch_fn, queries: dict, top_k: int = 100) -> float:
    """Return queries-per-second (single warm-up pass then timed pass)."""
    # warm-up
    warmup_q = dict(list(queries.items())[:5])
    _ = retrieve_batch_fn(warmup_q, top_k=top_k)
    # timed
    t0 = time.time()
    _ = retrieve_batch_fn(queries, top_k=top_k)
    elapsed = time.time() - t0
    return len(queries) / elapsed, elapsed


print("Measuring query throughput on the 323 test queries (CPU)...")
qps_bm25,    t_bm25_tp    = _throughput(bm25_retrieve_batch,              test_queries, TOP_K_RETRIEVE)
qps_minilm,  t_minilm_tp  = _throughput(dense_retrieve_batch,             test_queries, TOP_K_RETRIEVE)
qps_bge,     t_bge_tp     = _throughput(bge_retriever["retrieve_batch"],  test_queries, TOP_K_RETRIEVE)
qps_e5,      t_e5_tp      = _throughput(e5_retriever["retrieve_batch"],   test_queries, TOP_K_RETRIEVE)

# Embedding matrix sizes (in MB)
_emb_minilm_mb = doc_embeddings.nbytes / 1024**2
_emb_bge_mb    = bge_retriever["embeddings"].nbytes / 1024**2
_emb_e5_mb     = e5_retriever["embeddings"].nbytes / 1024**2

eff_rows = [
    {
        "Retriever":           "BM25",
        "Index build (s)":     f"{bm25_index_time:.1f}",
        "Corpus encode (s)":   "—",
        "Emb. matrix (MB)":    "— (inverted list)",
        "Throughput (q/s)":    f"{qps_bm25:.0f}",
        "Latency/query (ms)":  f"{1000*t_bm25_tp/len(test_queries):.1f}",
    },
    {
        "Retriever":           "Dense (MiniLM-L6)",
        "Index build (s)":     f"{dense_encode_time:.1f}",
        "Corpus encode (s)":   f"{dense_encode_time:.1f}",
        "Emb. matrix (MB)":    f"{_emb_minilm_mb:.0f}",
        "Throughput (q/s)":    f"{qps_minilm:.0f}",
        "Latency/query (ms)":  f"{1000*t_minilm_tp/len(test_queries):.1f}",
    },
    {
        "Retriever":           "Dense (BGE-small)",
        "Index build (s)":     f"{bge_retriever['encode_time']:.1f}",
        "Corpus encode (s)":   f"{bge_retriever['encode_time']:.1f}",
        "Emb. matrix (MB)":    f"{_emb_bge_mb:.0f}",
        "Throughput (q/s)":    f"{qps_bge:.0f}",
        "Latency/query (ms)":  f"{1000*t_bge_tp/len(test_queries):.1f}",
    },
    {
        "Retriever":           "Dense (E5-small)",
        "Index build (s)":     f"{e5_retriever['encode_time']:.1f}",
        "Corpus encode (s)":   f"{e5_retriever['encode_time']:.1f}",
        "Emb. matrix (MB)":    f"{_emb_e5_mb:.0f}",
        "Throughput (q/s)":    f"{qps_e5:.0f}",
        "Latency/query (ms)":  f"{1000*t_e5_tp/len(test_queries):.1f}",
    },
]

eff_df = _pd.DataFrame(eff_rows)
print()
print("Efficiency Summary")
print("=" * 85)
print(eff_df.to_string(index=False))
print("=" * 85)
print()
print("Notes:")
print("  • All timings are single-threaded CPU.")
print("  • 'Emb. matrix (MB)' = float32 embedding matrix; FAISS overhead is ~10%.")
print("  • Latency/query is amortised over the full 323-query batch.")
print("  • BM25 index_memory_mb is dominated by the Python list of token lists.")
print("  • Index memory delta is reported by the build_dense_retriever function.")
print()
print(f"  BGE-small process memory delta: {bge_retriever['index_memory_mb']:.0f} MB")
print(f"  E5-small  process memory delta: {e5_retriever['index_memory_mb']:.0f} MB")


In [ ]:
print("Evaluating retrieval quality...\n")

# Evaluate each method
evaluations = {}
evaluations["BM25"] = evaluate_retriever(bm25_results, test_qrels)
evaluations["Dense"] = evaluate_retriever(dense_results, test_qrels)


# ── Print comparison table ───────────────────────────────────
method_names = list(evaluations.keys())

print("=" * (30 + 15 * len(method_names)))
print(f"{'Metric':<30}" + "".join(f"{name:<15}" for name in method_names))
print("=" * (30 + 15 * len(method_names)))

# Decide which metrics to display
display_metrics = []
for k in CONFIG["top_k_eval"]:
    display_metrics.append(f"nDCG@{k}")
display_metrics.append("---")
for k in CONFIG["top_k_eval"]:
    display_metrics.append(f"MAP@{k}")
display_metrics.append("---")
display_metrics.extend(["Recall@10", "P@10"])
display_metrics.append("---")
display_metrics.extend(["empirical_risk", "loss_variance", "loss_std", "n_queries"])

for metric in display_metrics:
    if metric == "---":
        print("-" * (30 + 15 * len(method_names)))
        continue

    row = f"{metric:<30}"
    for name in method_names:
        val = evaluations[name]["aggregate"].get(metric, "N/A")
        if isinstance(val, float):
            row += f"{val:<15.4f}"
        else:
            row += f"{str(val):<15}"
    print(row)

print("=" * (30 + 15 * len(method_names)))

# ── New dense baselines ───────────────────────────────────────────────────
evaluations["BGE-small"] = evaluate_retriever(bge_results, test_qrels)
evaluations["E5-small"]  = evaluate_retriever(e5_results,  test_qrels)
method_names = list(evaluations.keys())  # all 4 retrievers

# Quick summary for the new models
print("\n  Extended summary (all dense baselines):")
for _name in ["BGE-small", "E5-small"]:
    _agg = evaluations[_name]["aggregate"]
    print(f"  {_name:<12}  nDCG@10={_agg['nDCG@10']:.4f}  "
          f"MAP@10={_agg['MAP@10']:.4f}  "
          f"Recall@10={_agg['Recall@10']:.4f}")


In [ ]:
# ============================================================================
# P3 (extended): SPLADE and MedCPT retrievers
#
# SPLADE (naver/splade-cocondenser-ensembledistil):
#   Learned sparse retrieval — BERT MLM head + log(1+ReLU) activation.
#   Representations are vocab-sized sparse vectors; retrieval via dot product.
#   Better generalisation than BM25 (still interpretable in token space).
#
# MedCPT (ncbi/MedCPT-Query-Encoder + ncbi/MedCPT-Article-Encoder):
#   Dual-encoder dense retriever trained on PubMed click-through data.
#   Separate query / article encoders (768-dim CLS, L2-normalised).
#   Expected to outperform generic encoders on biomedical text.
# ============================================================================
import time as _time_sm
import os as _os_sm
import numpy as np
import torch
import torch.nn.functional as F
import scipy.sparse as sp
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModel
from tqdm.auto import tqdm

_device_sm = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {_device_sm}")
print()

# ── Memory helper (reuse _psutil if available, else skip) ─────────────────
try:
    import psutil as _psutil_sm
    _proc_sm = _psutil_sm.Process(_os_sm.getpid())
    def _mem_mb(): return _proc_sm.memory_info().rss / 1024**2
except ImportError:
    def _mem_mb(): return float("nan")


# ═════════════════════════════════════════════════════════════════════════════
# SPLADE
# ═════════════════════════════════════════════════════════════════════════════

SPLADE_MODEL_NAME = "naver/splade-cocondenser-ensembledistil"


def _splade_encode(
    texts: list,
    tokenizer,
    model,
    batch_size: int = 32,
    max_length: int = 256,
    desc: str = "SPLADE",
) -> sp.csr_matrix:
    """
    Encode a list of texts to SPLADE sparse vectors.
    Returns a scipy CSR matrix of shape (len(texts), vocab_size).
    Activation: max_token[ log(1 + ReLU(logit)) ].
    """
    rows = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc, leave=False):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )
        inputs = {k: v.to(_device_sm) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs)
        # (B, T, V) → apply SPLADE activation, mask padding, max-pool over T
        vecs = torch.log(1 + torch.relu(out.logits))  # (B, T, V)
        vecs = (
            vecs * inputs["attention_mask"].unsqueeze(-1)
        ).max(dim=1).values  # (B, V)
        rows.append(sp.csr_matrix(vecs.cpu().numpy()))
    return sp.vstack(rows)


print(f"Loading SPLADE model: {SPLADE_MODEL_NAME} ...")
_splade_tok = AutoTokenizer.from_pretrained(SPLADE_MODEL_NAME)
_splade_mod = (
    AutoModelForMaskedLM.from_pretrained(SPLADE_MODEL_NAME).to(_device_sm).eval()
)

_m0 = _mem_mb()
_t0 = _time_sm.time()
_splade_doc_vecs = _splade_encode(
    doc_texts, _splade_tok, _splade_mod, desc="SPLADE corpus"
)
_splade_encode_time = _time_sm.time() - _t0
_splade_mem_mb = _mem_mb() - _m0

print(
    f"  Built in {_splade_encode_time:.1f}s"
    f"  |  sparse matrix {_splade_doc_vecs.shape}"
    f"  |  avg nnz/doc = {_splade_doc_vecs.nnz / _splade_doc_vecs.shape[0]:.0f}"
    f"  |  memory delta ≈ {_splade_mem_mb:.0f} MB"
)


def _splade_retrieve_batch(queries: dict, top_k: int = 100) -> dict:
    q_texts = list(queries.values())
    q_vecs = _splade_encode(q_texts, _splade_tok, _splade_mod, desc="SPLADE qry")
    # (n_q, n_docs)  — sparse × sparse.T → dense
    scores = (q_vecs @ _splade_doc_vecs.T).toarray()
    results = {}
    _k = min(top_k, len(doc_ids))
    for i, qid in enumerate(queries.keys()):
        top_idx = np.argpartition(-scores[i], _k)[:_k]
        top_idx = top_idx[np.argsort(-scores[i][top_idx])]
        results[qid] = {doc_ids[j]: float(scores[i][j]) for j in top_idx}
    return results


print("✓ SPLADE ready.")
print()


# ═════════════════════════════════════════════════════════════════════════════
# MedCPT  (separate query + article encoders)
# ═════════════════════════════════════════════════════════════════════════════

MEDCPT_QUERY_MODEL   = "ncbi/MedCPT-Query-Encoder"
MEDCPT_ARTICLE_MODEL = "ncbi/MedCPT-Article-Encoder"


def _hf_encode(
    texts: list,
    tokenizer,
    model,
    batch_size: int = 32,
    max_length: int = 512,
    desc: str = "HF encode",
) -> "np.ndarray":
    """
    Encode texts with a HuggingFace AutoModel; returns L2-normalised
    CLS-token embeddings as float32 numpy array (n, dim).
    """
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc, leave=False):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )
        inputs = {k: v.to(_device_sm) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs)
        embs = F.normalize(out.last_hidden_state[:, 0, :], dim=-1)  # CLS + L2
        all_embs.append(embs.cpu().numpy())
    return np.concatenate(all_embs, axis=0).astype("float32")


print(f"Loading MedCPT article encoder: {MEDCPT_ARTICLE_MODEL} ...")
_medcpt_art_tok = AutoTokenizer.from_pretrained(MEDCPT_ARTICLE_MODEL)
_medcpt_art_mod = AutoModel.from_pretrained(MEDCPT_ARTICLE_MODEL).to(_device_sm).eval()

_m0 = _mem_mb()
_t0 = _time_sm.time()
_medcpt_doc_embs = _hf_encode(
    doc_texts,
    _medcpt_art_tok,
    _medcpt_art_mod,
    max_length=512,
    desc="MedCPT corpus",
)
_medcpt_encode_time = _time_sm.time() - _t0
_medcpt_mem_mb = _mem_mb() - _m0

_medcpt_dim   = _medcpt_doc_embs.shape[1]
_medcpt_index = faiss.IndexFlatIP(_medcpt_dim)
_medcpt_index.add(_medcpt_doc_embs)
print(
    f"  Article encoder: {_medcpt_encode_time:.1f}s"
    f"  |  dim={_medcpt_dim}"
    f"  |  memory delta ≈ {_medcpt_mem_mb:.0f} MB"
)

print(f"Loading MedCPT query encoder: {MEDCPT_QUERY_MODEL} ...")
_medcpt_qry_tok = AutoTokenizer.from_pretrained(MEDCPT_QUERY_MODEL)
_medcpt_qry_mod = AutoModel.from_pretrained(MEDCPT_QUERY_MODEL).to(_device_sm).eval()
print("  Query encoder loaded.")
print()


def _medcpt_retrieve_batch(queries: dict, top_k: int = 100) -> dict:
    q_texts = list(queries.values())
    q_embs  = _hf_encode(
        q_texts,
        _medcpt_qry_tok,
        _medcpt_qry_mod,
        max_length=64,   # MedCPT query encoder was trained with max_length=64
        desc="MedCPT qry",
    )
    scores_arr, indices_arr = _medcpt_index.search(q_embs, top_k)
    results = {}
    for i, qid in enumerate(queries.keys()):
        results[qid] = {}
        for j in range(top_k):
            idx = indices_arr[i][j]
            if 0 <= idx < len(doc_ids):
                results[qid][doc_ids[idx]] = float(scores_arr[i][j])
    return results


print("✓ MedCPT ready.")
print()


# ═════════════════════════════════════════════════════════════════════════════
# Retrieval on test set
# ═════════════════════════════════════════════════════════════════════════════

print("Running SPLADE retrieval on test set...")
_t0 = _time_sm.time()
splade_results = _splade_retrieve_batch(test_queries, top_k=TOP_K_RETRIEVE)
t_splade = _time_sm.time() - _t0
print(f"  Done in {t_splade:.2f}s  ({len(splade_results)} queries)")

print("Running MedCPT retrieval on test set...")
_t0 = _time_sm.time()
medcpt_results = _medcpt_retrieve_batch(test_queries, top_k=TOP_K_RETRIEVE)
t_medcpt = _time_sm.time() - _t0
print(f"  Done in {t_medcpt:.2f}s  ({len(medcpt_results)} queries)")
print()


# ═════════════════════════════════════════════════════════════════════════════
# Evaluate + register in global evaluations dict
# ═════════════════════════════════════════════════════════════════════════════

evaluations["SPLADE"] = evaluate_retriever(splade_results, test_qrels)
evaluations["MedCPT"] = evaluate_retriever(medcpt_results, test_qrels)
method_names = list(evaluations.keys())

COLORS["SPLADE"] = "#1abc9c"
COLORS["MedCPT"] = "#e67e22"


# ═════════════════════════════════════════════════════════════════════════════
# Summary comparison table (all retrievers)
# ═════════════════════════════════════════════════════════════════════════════

# Gather timing for throughput column
_timing = {
    "BM25":      t_bm25,
    "Dense":     t_dense,
    "BGE-small": t_bge,
    "E5-small":  t_e5,
    "SPLADE":    t_splade,
    "MedCPT":    t_medcpt,
}

print("=" * 78)
print(
    f"  {'Retriever':<16}  {'nDCG@10':>8}  {'MAP@10':>7}  "
    f"{'Recall@10':>9}  {'P@10':>6}  {'q/s':>6}"
)
print("-" * 78)
for _n in method_names:
    _a   = evaluations[_n]["aggregate"]
    _qps = len(test_queries) / _timing.get(_n, float("nan"))
    print(
        f"  {_n:<16}  {_a['nDCG@10']:>8.4f}  {_a['MAP@10']:>7.4f}  "
        f"{_a['Recall@10']:>9.4f}  {_a.get('P@10', float('nan')):>6.4f}  {_qps:>6.0f}"
    )
print("=" * 78)
print()
print(f"  SPLADE encode time: {_splade_encode_time:.1f}s"
      f"  |  avg nnz/doc: {_splade_doc_vecs.nnz / _splade_doc_vecs.shape[0]:.0f}"
      f"  |  memory delta: {_splade_mem_mb:.0f} MB")
print(f"  MedCPT encode time: {_medcpt_encode_time:.1f}s"
      f"  |  dim: {_medcpt_dim}"
      f"  |  memory delta: {_medcpt_mem_mb:.0f} MB")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: nDCG at different k values ──────────────────────
k_values = CONFIG["top_k_eval"]
for name in method_names:
    ndcg_values = [evaluations[name]["aggregate"][f"nDCG@{k}"] for k in k_values]
    axes[0].plot(k_values, ndcg_values, "o-", label=name,
                 color=COLORS.get(name, "gray"), linewidth=2, markersize=8)

axes[0].set_xlabel("k (cutoff)")
axes[0].set_ylabel("nDCG@k")
axes[0].set_title("nDCG at Different Cutoffs")
axes[0].legend()
axes[0].set_xticks(k_values)

# ── Plot 2: Grouped bar chart of key metrics ────────────────
key_metrics = ["nDCG@10", "MAP@10", "Recall@10", "P@10"]
x = np.arange(len(key_metrics))
width = 0.25

for i, name in enumerate(method_names):
    values = [evaluations[name]["aggregate"][m] for m in key_metrics]
    bars = axes[1].bar(x + i * width, values, width, label=name,
                       color=COLORS.get(name, "gray"), alpha=0.8)
    # Add value labels
    for bar, val in zip(bars, values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f"{val:.3f}", ha="center", fontsize=8)

axes[1].set_xlabel("Metric")
axes[1].set_ylabel("Score")
axes[1].set_title("Key Metrics Comparison")
axes[1].set_xticks(x + width * (len(method_names) - 1) / 2)
axes[1].set_xticklabels(key_metrics)
axes[1].legend()
axes[1].set_ylim(0, max(0.5, axes[1].get_ylim()[1] * 1.1))

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["results_dir"], "metric_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

In [ ]:
from scipy import stats

print("Per-Query Loss Distribution Statistics")
print("=" * 65)
print(f"{'Statistic':<25}" + "".join(f"{name:<15}" for name in method_names))
print("-" * 65)

loss_stats = {}
for name in method_names:
    losses = evaluations[name]["loss_array"]
    loss_stats[name] = {
        "mean (R̂)": np.mean(losses),
        "median": np.median(losses),
        "std (σ)": np.std(losses, ddof=1),
        "variance (σ²)": np.var(losses, ddof=1),
        "min": np.min(losses),
        "max": np.max(losses),
        "Q25": np.percentile(losses, 25),
        "Q75": np.percentile(losses, 75),
        "IQR": np.percentile(losses, 75) - np.percentile(losses, 25),
        "skewness": float(stats.skew(losses)),
        "kurtosis": float(stats.kurtosis(losses)),
        "% total fail (ℓ=1)": np.mean(losses >= 0.99) * 100,
        "% perfect (ℓ=0)": np.mean(losses <= 0.01) * 100,
    }

for stat_name in loss_stats[method_names[0]]:
    row = f"{stat_name:<25}"
    for name in method_names:
        val = loss_stats[name][stat_name]
        row += f"{val:<15.4f}"
    print(row)

print("=" * 65)

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Plot 1: Overlapping histograms ──────────────────────────
ax = axes[0]
for name in method_names:
    losses = evaluations[name]["loss_array"]
    ax.hist(losses, bins=30, alpha=0.5, label=name,
            color=COLORS.get(name, "gray"), density=True, edgecolor="white")
ax.set_xlabel("Per-Query Loss (1 - nDCG@10)")
ax.set_ylabel("Density")
ax.set_title("Loss Distribution Comparison")
ax.legend()
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.5)

# ── Plot 2: Box plots ───────────────────────────────────────
ax = axes[1]
loss_data = [evaluations[name]["loss_array"] for name in method_names]
bp = ax.boxplot(loss_data, labels=method_names, patch_artist=True,
                widths=0.5, showmeans=True,
                meanprops=dict(marker="D", markerfacecolor="black", markersize=6))
for patch, name in zip(bp["boxes"], method_names):
    patch.set_facecolor(COLORS.get(name, "gray"))
    patch.set_alpha(0.6)
ax.set_ylabel("Per-Query Loss")
ax.set_title("Loss Distribution (Box Plot)")

# ── Plot 3: Empirical CDF ───────────────────────────────────
ax = axes[2]
for name in method_names:
    losses = np.sort(evaluations[name]["loss_array"])
    cdf = np.arange(1, len(losses) + 1) / len(losses)
    ax.plot(losses, cdf, label=name, color=COLORS.get(name, "gray"), linewidth=2)
ax.set_xlabel("Per-Query Loss")
ax.set_ylabel("Cumulative Probability")
ax.set_title("Empirical CDF of Losses")
ax.legend()
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["results_dir"], "loss_distributions.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

---
## 4. Generation Quality (RAG Completion)

Retrieval alone is only half of RAG.  Here we close the loop by passing the
**top-3 retrieved documents** as context to **`google/flan-t5-base`** (~250 M
parameters, free to run locally via HuggingFace `transformers`).

### Reference answers
NFCorpus has no pre-written gold answers — it is a retrieval benchmark.  We
therefore use the **concatenated text of all documents with qrel relevance ≥ 2**
as the oracle reference.  This is a standard RAG-evaluation workaround for
retrieval-only corpora: a retriever that surfaces better evidence should enable
the generator to produce text more similar to that evidence.

### Metric
We report **ROUGE-1 / ROUGE-2 / ROUGE-L** (F1, macro-averaged over test
queries) with `rouge-score`.  A higher ROUGE score means the generator, when
given retrieved context, reproduced more of the oracle-relevant text.


In [ ]:
from transformers import pipeline
from rouge_score import rouge_scorer as rouge_scorer_lib

GEN_CONFIG = {
    "model":            "google/flan-t5-base",
    "top_k_context":    3,     # number of retrieved docs to use as context
    "max_chars_per_doc": 400,  # characters to keep per document snippet
    "max_new_tokens":   128,
    "min_relevance":    2,     # minimum qrel score to count as oracle answer
    "eval_queries":     50,    # test-query sample size (speed vs. stability)
}

print(f"Loading generator: {GEN_CONFIG['model']} …", end=" ", flush=True)
_t0 = time.time()
generator = pipeline(
    "text2text-generation",
    model=GEN_CONFIG["model"],
    device=-1,          # CPU; set to 0 if a CUDA GPU is available
    max_new_tokens=GEN_CONFIG["max_new_tokens"],
)
print(f"done ({time.time() - _t0:.1f} s)")

rouge_scorer = rouge_scorer_lib.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)
print("ROUGE scorer ready.")


In [ ]:
def build_context(retrieved: dict, corpus: dict,
                  top_k: int = 3, max_chars: int = 400) -> str:
    """Concatenate text snippets from top-k retrieved documents."""
    parts = []
    for doc_id in list(retrieved)[:top_k]:
        doc = corpus.get(doc_id, {})
        snippet = (doc.get("title", "") + ". " + doc.get("text", "")).strip()
        parts.append(snippet[:max_chars])
    return "\n\n".join(parts)


def make_prompt(query: str, context: str) -> str:
    return (
        "Answer the following question using only the provided context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )


def oracle_reference(qid: str, qrels: dict, corpus: dict,
                     min_rel: int = 2, max_chars: int = 2000) -> str:
    """Return concatenated text of all oracle-relevant docs for a query."""
    texts = [
        (corpus[did].get("title", "") + ". " + corpus[did].get("text", "")).strip()
        for did, rel in qrels.get(qid, {}).items()
        if rel >= min_rel and did in corpus
    ]
    return " ".join(texts)[:max_chars]


def score_rouge(hypothesis: str, reference: str, scorer) -> dict:
    if not reference.strip() or not hypothesis.strip():
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    s = scorer.score(reference, hypothesis)
    return {k: s[k].fmeasure for k in ("rouge1", "rouge2", "rougeL")}


In [ ]:
# Select test queries that have at least one oracle-relevant document
gen_qids = [
    qid for qid in test_queries
    if any(
        rel >= GEN_CONFIG["min_relevance"]
        for rel in test_qrels.get(qid, {}).values()
    )
][: GEN_CONFIG["eval_queries"]]
print(f"Evaluating RAG generation on {len(gen_qids)} test queries …\n")

retrieval_sources = {"BM25": bm25_results, "Dense": dense_results}
gen_rouge: dict[str, dict[str, list]] = {
    name: {"rouge1": [], "rouge2": [], "rougeL": []}
    for name in retrieval_sources
}

for method_name, ret_results in retrieval_sources.items():
    print(f"  [{method_name}] generating …", end=" ", flush=True)
    _t0 = time.time()
    for qid in gen_qids:
        context  = build_context(
            ret_results.get(qid, {}), corpus,
            top_k=GEN_CONFIG["top_k_context"],
            max_chars=GEN_CONFIG["max_chars_per_doc"],
        )
        prompt    = make_prompt(test_queries[qid], context)
        generated = generator(prompt)[0]["generated_text"]
        reference = oracle_reference(
            qid, test_qrels, corpus, min_rel=GEN_CONFIG["min_relevance"]
        )
        rouge = score_rouge(generated, reference, rouge_scorer)
        for metric in ("rouge1", "rouge2", "rougeL"):
            gen_rouge[method_name][metric].append(rouge[metric])
    print(f"done in {time.time() - _t0:.1f} s")

# Aggregate
gen_summary: dict[str, dict[str, float]] = {
    name: {m: float(np.mean(v)) for m, v in scores.items()}
    for name, scores in gen_rouge.items()
}

print("\nGeneration Quality — ROUGE F1 (macro-avg, n={})".format(len(gen_qids)))
print("=" * 48)
header = f"{'Metric':<12}" + "".join(f"{n:<14}" for n in gen_summary)
print(header)
print("-" * 48)
metric_labels = {"rouge1": "ROUGE-1", "rouge2": "ROUGE-2", "rougeL": "ROUGE-L"}
for key, label in metric_labels.items():
    row = f"{label:<12}" + "".join(
        f"{gen_summary[n][key]:<14.4f}" for n in gen_summary
    )
    print(row)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rouge_keys   = ["rouge1", "rouge2", "rougeL"]
rouge_labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
x = np.arange(len(rouge_keys))
bar_width = 0.35

# ── Left: grouped bar chart ──────────────────────────────────
ax = axes[0]
for i, (name, scores) in enumerate(gen_summary.items()):
    vals = [scores[k] for k in rouge_keys]
    offset = (i - (len(gen_summary) - 1) / 2) * bar_width
    bars = ax.bar(
        x + offset, vals, bar_width,
        label=name, color=COLORS.get(name, "gray"), alpha=0.85, edgecolor="white",
    )
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f"{val:.3f}", ha="center", va="bottom", fontsize=8.5,
        )

ax.set_xticks(x)
ax.set_xticklabels(rouge_labels, fontsize=11)
ax.set_ylabel("ROUGE F1")
ax.set_title(
    f"RAG Generation Quality\n"
    f"(flan-t5-base · top-{GEN_CONFIG['top_k_context']} docs · n={len(gen_qids)} queries)"
)
ax.legend()
ax.set_ylim(0, max(
    scores[k] for scores in gen_summary.values() for k in rouge_keys
) * 1.25)
ax.grid(axis="y", alpha=0.3)

# ── Right: retrieval nDCG@10 vs ROUGE-L scatter per query ────
ax2 = axes[1]
for name, color in COLORS.items():
    if name not in gen_rouge:
        continue
    ndcg_vals = [
        compute_ndcg(
            list(retrieval_sources[name].get(qid, {}).keys()),
            test_qrels.get(qid, {}),
            k=10,
        )
        for qid in gen_qids
    ]
    ax2.scatter(
        ndcg_vals, gen_rouge[name]["rougeL"],
        alpha=0.4, s=25, color=color, label=name,
    )
    # linear trend
    z = np.polyfit(ndcg_vals, gen_rouge[name]["rougeL"], 1)
    xline = np.linspace(min(ndcg_vals), max(ndcg_vals), 100)
    ax2.plot(xline, np.polyval(z, xline), color=color, linewidth=1.8)

ax2.set_xlabel("nDCG@10 (retrieval quality)")
ax2.set_ylabel("ROUGE-L (generation quality)")
ax2.set_title("Retrieval Quality → Generation Quality\n(per-query, with trend)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
_fig_path = os.path.join(CONFIG["results_dir"], "generation_quality.png")
plt.savefig(_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {_fig_path}")


---
## 5. Generalization Bounds

This is the core theoretical contribution. We compute three types of bounds
that answer: **"How confident can we be that the observed performance 
generalizes to unseen queries?"**

### 5.1 Hoeffding Bound (distribution-free)
$$R(h) \leq \hat{R}_n(h) + \sqrt{\frac{\ln(2/\delta)}{2n}}$$
Same for all methods — cannot distinguish inductive biases.

### 5.2 Empirical Bernstein Bound (variance-aware) ⭐
$$R(h) \leq \hat{R}_n(h) + \sqrt{\frac{2\hat{\sigma}^2\ln(3/\delta)}{n}} + \frac{3\ln(3/\delta)}{n-1}$$
**KEY BOUND**: tighter when variance is lower → rewards consistent methods.

### 5.3 Bootstrap Confidence Interval (non-parametric)
Resampling-based estimate of the uncertainty in R̂.


In [ ]:
# ============================================================================
# CELL 23 — Generalization Bound Functions
# ============================================================================

def hoeffding_bound(loss_array: np.ndarray, delta: float = 0.05) -> dict:
    n = len(loss_array)
    R_hat = np.mean(loss_array)
    epsilon = np.sqrt(np.log(2.0 / delta) / (2 * n))

    return {
        "bound_type": "Hoeffding",
        "R_hat": R_hat,
        "epsilon": epsilon,
        "upper_bound": R_hat + epsilon,
        "n": n,
        "delta": delta,
    }


def bernstein_bound(loss_array: np.ndarray, delta: float = 0.05) -> dict:
    n = len(loss_array)
    R_hat = np.mean(loss_array)
    sigma_sq = np.var(loss_array, ddof=1)

    ln_term = np.log(3.0 / delta)
    variance_term = np.sqrt(2 * sigma_sq * ln_term / n)
    remainder_term = 3 * ln_term / (n - 1)
    epsilon = variance_term + remainder_term

    return {
        "bound_type": "Bernstein",
        "R_hat": R_hat,
        "epsilon": epsilon,
        "upper_bound": R_hat + epsilon,
        "sigma_sq": sigma_sq,
        "variance_term": variance_term,
        "remainder_term": remainder_term,
        "n": n,
        "delta": delta,
    }


def bootstrap_ci(
    loss_array: np.ndarray,
    n_bootstrap: int = 10000,
    alpha: float = 0.05,
    seed: int = SEED,
) -> dict:
    rng = np.random.RandomState(seed)
    n = len(loss_array)
    R_hat = np.mean(loss_array)

    bootstrap_means = np.array([
        np.mean(rng.choice(loss_array, size=n, replace=True))
        for _ in range(n_bootstrap)
    ])

    ci_lower = np.percentile(bootstrap_means, 100 * alpha / 2)
    ci_upper = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))

    return {
        "bound_type": "Bootstrap",
        "R_hat": R_hat,
        "epsilon": ci_upper - R_hat,
        "upper_bound": ci_upper,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "ci_width": ci_upper - ci_lower,
        "bootstrap_std": np.std(bootstrap_means),
        "n": n,
        "delta": alpha,
    }


print("✓ Bound functions defined.")

In [ ]:
delta = CONFIG["delta"]
all_bounds = {}

print(f"Computing generalization bounds (δ = {delta}, confidence = {(1-delta)*100:.0f}%)")
print()

for name in method_names:
    losses = evaluations[name]["loss_array"]
    all_bounds[name] = {
        "Hoeffding": hoeffding_bound(losses, delta),
        "Bernstein": bernstein_bound(losses, delta),
        "Bootstrap": bootstrap_ci(losses, CONFIG["n_bootstrap"], delta),
    }

# ── Print comprehensive bound comparison ─────────────────────
print("=" * (35 + 18 * len(method_names)))
print(f"{'Generalization Bounds':<35}" + "".join(f"{name:<18}" for name in method_names))
print("=" * (35 + 18 * len(method_names)))

rows = [
    ("Empirical Risk R̂(h)", lambda b: b["Hoeffding"]["R_hat"]),
    ("Per-query variance σ²", lambda b: b["Bernstein"]["sigma_sq"]),
    ("Per-query std dev σ", lambda b: np.sqrt(b["Bernstein"]["sigma_sq"])),
    ("", None),  # separator
    ("--- Epsilon (bound width) ---", None),
    ("ε Hoeffding", lambda b: b["Hoeffding"]["epsilon"]),
    ("ε Bernstein", lambda b: b["Bernstein"]["epsilon"]),
    ("ε Bootstrap", lambda b: b["Bootstrap"]["epsilon"]),
    ("", None),
    ("--- True Risk Upper Bound ---", None),
    ("R(h) ≤  (Hoeffding)", lambda b: b["Hoeffding"]["upper_bound"]),
    ("R(h) ≤  (Bernstein)", lambda b: b["Bernstein"]["upper_bound"]),
    ("R(h) ≤  (Bootstrap)", lambda b: b["Bootstrap"]["upper_bound"]),
    ("", None),
    ("--- Bootstrap CI ---", None),
    ("CI lower", lambda b: b["Bootstrap"]["ci_lower"]),
    ("CI upper", lambda b: b["Bootstrap"]["ci_upper"]),
    ("CI width", lambda b: b["Bootstrap"]["ci_width"]),
]

for label, fn in rows:
    if fn is None:
        print(f"  {label}")
        continue
    row = f"  {label:<33}"
    for name in method_names:
        val = fn(all_bounds[name])
        row += f"{val:<18.4f}"
    print(row)

print("=" * (35 + 18 * len(method_names)))

# ── Interpretation ───────────────────────────────────────────
bern_epsilons = {name: all_bounds[name]["Bernstein"]["epsilon"] for name in method_names}
tightest = min(bern_epsilons, key=bern_epsilons.get)
loosest = max(bern_epsilons, key=bern_epsilons.get)

print(f"\nInterpretation:")
print(f"  → {tightest} has the TIGHTEST Bernstein bound (ε={bern_epsilons[tightest]:.4f})")
print(f"    Its performance estimate is most reliable for unseen queries.")
print(f"  → {loosest} has the LOOSEST Bernstein bound (ε={bern_epsilons[loosest]:.4f})")
print(f"    More uncertainty about its true performance.")
print(f"  → Hoeffding ε is identical for all methods ({all_bounds[method_names[0]]['Hoeffding']['epsilon']:.4f})")
print(f"    confirming it cannot distinguish inductive biases.")

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
x_pos = np.arange(len(method_names))

for i, name in enumerate(method_names):
    R_hat = all_bounds[name]["Bernstein"]["R_hat"]
    ci_lower = all_bounds[name]["Bootstrap"]["ci_lower"]
    ci_upper = all_bounds[name]["Bootstrap"]["ci_upper"]
    bern_upper = all_bounds[name]["Bernstein"]["upper_bound"]

    # Bootstrap CI
    ax.errorbar(i, R_hat, yerr=[[R_hat - ci_lower], [ci_upper - R_hat]],
                fmt="o", color=COLORS.get(name, "gray"), capsize=8,
                capthick=2, markersize=10, linewidth=2,
                label=f"{name} (Bootstrap 95% CI)")

    # Bernstein upper bound (triangle marker)
    ax.scatter(i + 0.15, bern_upper, marker="v", s=100,
               color=COLORS.get(name, "gray"), zorder=5, edgecolors="black")

ax.set_xticks(x_pos)
ax.set_xticklabels(method_names)
ax.set_ylabel("Risk (1 - nDCG@10)")
ax.set_title("Empirical Risk with Generalization Bounds\n(dots=R̂, bars=Bootstrap CI, ▼=Bernstein upper)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)

# ── Plot 2: Epsilon comparison ───────────────────────────────
ax = axes[1]
bound_types = ["Hoeffding", "Bernstein", "Bootstrap"]
x = np.arange(len(bound_types))
width = 0.25

for i, name in enumerate(method_names):
    epsilons = [all_bounds[name][bt]["epsilon"] for bt in bound_types]
    ax.bar(x + i * width, epsilons, width, label=name,
           color=COLORS.get(name, "gray"), alpha=0.8, edgecolor="white")

ax.set_xticks(x + width * (len(method_names) - 1) / 2)
ax.set_xticklabels(bound_types)
ax.set_ylabel("ε (bound width)")
ax.set_title("Generalization Bound Tightness\n(smaller ε = stronger guarantee)")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["results_dir"], "generalization_bounds.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

---
## 6. Train-Test Generalization Gap
The generalization gap measures how well performance on "seen" queries
predicts performance on "unseen" queries:
$$G(h) = R_{test}(h) - R_{train}(h)$$
- **Positive gap**: test is harder than train (expected)
- **Large gap**: the method may be overfitting to training query patterns
- **Small gap**: the method generalizes well
We test statistical significance using the Mann-Whitney U test.


In [ ]:

print("Computing train-test generalization gap...\n")

train_bm25 = bm25_retrieve_batch(train_queries, top_k=TOP_K_RETRIEVE)
train_dense = dense_retrieve_batch(train_queries, top_k=TOP_K_RETRIEVE)

train_retrievals = {"BM25": train_bm25, "Dense": train_dense}

train_evaluations = {}
for name, results in train_retrievals.items():
    train_evaluations[name] = evaluate_retriever(results, train_qrels)

gap_results = {}
for name in method_names:
    train_losses = train_evaluations[name]["loss_array"]
    test_losses = evaluations[name]["loss_array"]

    R_train = np.mean(train_losses)
    R_test = np.mean(test_losses)
    gap = R_test - R_train

    # Mann-Whitney U test (non-parametric)
    stat, p_value = stats.mannwhitneyu(
        train_losses, test_losses, alternative="two-sided"
    )

    gap_results[name] = {
        "R_train": R_train,
        "R_test": R_test,
        "gap": gap,
        "abs_gap": abs(gap),
        "p_value": p_value,
        "significant": p_value < CONFIG["delta"],
        "train_var": np.var(train_losses, ddof=1),
        "test_var": np.var(test_losses, ddof=1),
        "n_train": len(train_losses),
        "n_test": len(test_losses),
        "train_losses": train_losses,
        "test_losses": test_losses,
    }

# ── Print results ────────────────────────────────────────────
print("=" * (30 + 18 * len(method_names)))
print(f"{'Generalization Gap':<30}" + "".join(f"{name:<18}" for name in method_names))
print("=" * (30 + 18 * len(method_names)))

display_rows = [
    "R_train", "R_test", "gap", "abs_gap",
    "p_value", "significant",
    "train_var", "test_var",
    "n_train", "n_test",
]

for metric in display_rows:
    row = f"  {metric:<28}"
    for name in method_names:
        val = gap_results[name][metric]
        if isinstance(val, (float, np.floating)):
            row += f"{val:<18.4f}"
        else:
            row += f"{str(val):<18}"
    print(row)

print("=" * (30 + 18 * len(method_names)))

# Interpretation
smallest_gap = min(method_names, key=lambda n: gap_results[n]["abs_gap"])
print(f"\nInterpretation:")
print(f"  → {smallest_gap} has the smallest generalization gap "
      f"({gap_results[smallest_gap]['abs_gap']:.4f})")
print(f"    suggesting its inductive bias generalizes best across query distributions.")

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Train vs Test Risk ───────────────────────────────
ax = axes[0]
x_pos = np.arange(len(method_names))
width = 0.3

train_risks = [gap_results[n]["R_train"] for n in method_names]
test_risks = [gap_results[n]["R_test"] for n in method_names]

bars1 = ax.bar(x_pos - width/2, train_risks, width, label="Train R̂",
               color="#3498db", alpha=0.7, edgecolor="white")
bars2 = ax.bar(x_pos + width/2, test_risks, width, label="Test R̂",
               color="#e74c3c", alpha=0.7, edgecolor="white")

# Add gap annotation
for i, name in enumerate(method_names):
    gap = gap_results[name]["gap"]
    y_top = max(train_risks[i], test_risks[i])
    ax.annotate(f"Δ={gap:+.3f}", xy=(i, y_top + 0.01),
                ha="center", fontsize=9, fontweight="bold")

ax.set_xticks(x_pos)
ax.set_xticklabels(method_names)
ax.set_ylabel("Empirical Risk R̂(h)")
ax.set_title("Train vs Test Empirical Risk")
ax.legend()

# ── Plot 2: Loss distributions for train vs test ────────────
ax = axes[1]
data_for_violin = []
labels_for_violin = []
colors_for_violin = []

for name in method_names:
    data_for_violin.append(gap_results[name]["train_losses"])
    labels_for_violin.append(f"{name}\nTrain")
    colors_for_violin.append(COLORS.get(name, "gray"))

    data_for_violin.append(gap_results[name]["test_losses"])
    labels_for_violin.append(f"{name}\nTest")
    colors_for_violin.append(COLORS.get(name, "gray"))

bp = ax.boxplot(data_for_violin, labels=labels_for_violin,
                patch_artist=True, widths=0.5, showmeans=True,
                meanprops=dict(marker="D", markerfacecolor="black", markersize=5))

for i, patch in enumerate(bp["boxes"]):
    patch.set_facecolor(colors_for_violin[i])
    patch.set_alpha(0.4 if i % 2 == 0 else 0.7)  # lighter for train

ax.set_ylabel("Per-Query Loss")
ax.set_title("Train vs Test Loss Distributions")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["results_dir"], "generalization_gap.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

---
## 7. Head-to-Head Per-Query Comparison

Aggregate metrics can mask important patterns. Here we compare methods
at the per-query level to understand:
- On which queries does BM25 beat Dense? And vice versa?
- Are there systematic patterns in WHERE each method fails?
- Do the methods make correlated or independent errors?

This analysis directly reveals the INDUCTIVE BIAS: each method has
different failure modes.


In [ ]:
common_qids = sorted(
    set(evaluations["BM25"]["per_query_losses"].keys())
    & set(evaluations["Dense"]["per_query_losses"].keys())
)

bm25_losses_paired = np.array([evaluations["BM25"]["per_query_losses"][q] for q in common_qids])
dense_losses_paired = np.array([evaluations["Dense"]["per_query_losses"][q] for q in common_qids])

# Where does each method win?
bm25_wins = np.sum(bm25_losses_paired < dense_losses_paired)
dense_wins = np.sum(dense_losses_paired < bm25_losses_paired)
ties = np.sum(bm25_losses_paired == dense_losses_paired)
loss_diff = bm25_losses_paired - dense_losses_paired  # positive = dense is better

print(f"Head-to-Head: BM25 vs Dense ({len(common_qids)} queries)")
print(f"  BM25 wins:  {bm25_wins} ({bm25_wins/len(common_qids)*100:.1f}%)")
print(f"  Dense wins: {dense_wins} ({dense_wins/len(common_qids)*100:.1f}%)")
print(f"  Ties:       {ties} ({ties/len(common_qids)*100:.1f}%)")

# Paired statistical test
stat, p_paired = stats.wilcoxon(bm25_losses_paired, dense_losses_paired)
print(f"\n  Wilcoxon signed-rank test:")
print(f"    statistic = {stat:.1f}, p-value = {p_paired:.4f}")
print(f"    {'Significant' if p_paired < 0.05 else 'Not significant'} difference (α=0.05)")

# Correlation between methods
corr, p_corr = stats.pearsonr(bm25_losses_paired, dense_losses_paired)
print(f"\n  Loss correlation: r = {corr:.3f} (p = {p_corr:.4f})")
print(f"    {'High' if corr > 0.5 else 'Low'} correlation → ",
      f"{'similar' if corr > 0.5 else 'different'} failure modes")

# ── Paired bootstrap confidence intervals ──────────────────────────────────
# Bootstrap CI directly estimates the *mean* loss difference and its precision,
# which is the quantity that matters when comparing expected retrieval quality.
# Wilcoxon tests the *median rank-sum*: useful for null-hypothesis testing but
# provides no effect-size interval.
_n_bs = CONFIG["n_bootstrap"]
_rng_bs = np.random.default_rng(SEED)
_boot_mean_diffs = np.array(
    [np.mean(loss_diff[_rng_bs.integers(0, len(common_qids), size=len(common_qids))])
     for _ in range(_n_bs)]
)
_ci_lo, _ci_hi = np.percentile(_boot_mean_diffs, [2.5, 97.5])
_obs_mean = np.mean(loss_diff)

print(f"\n  Paired bootstrap ({_n_bs:,} resamples) — mean(BM25 loss − Dense loss):")
print(f"    Observed mean   : {_obs_mean:+.4f}")
print(f"    95% CI          : [{_ci_lo:+.4f}, {_ci_hi:+.4f}]")
if _ci_lo > 0:
    print("    → CI entirely above 0: Dense is significantly better (α=0.05).")
elif _ci_hi < 0:
    print("    → CI entirely below 0: BM25 is significantly better (α=0.05).")
else:
    print("    → CI straddles 0: difference is not significant at 95%.")
print()
print("  Bootstrap CI vs. Wilcoxon:")
print("    • Wilcoxon: tests whether the median signed rank differs from 0.")
print("    • Bootstrap CI: directly quantifies the mean nDCG@10 gap with")
print("      uncertainty bounds — an effect-size measure the Wilcoxon lacks.")
print("      This is the standard in modern IR evaluation (TREC, CLEF, etc.).")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Plot 1: Scatter — BM25 loss vs Dense loss per query ─────
ax = axes[0]
ax.scatter(bm25_losses_paired, dense_losses_paired, alpha=0.4, s=25,
           color="#34495e", edgecolors="white", linewidth=0.5)
ax.plot([0, 1], [0, 1], "r--", alpha=0.5, label="Equal performance")
ax.set_xlabel("BM25 Loss (1 - nDCG@10)")
ax.set_ylabel("Dense Loss (1 - nDCG@10)")
ax.set_title(f"Per-Query Loss Comparison\n(r={corr:.3f})")
ax.legend()
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# Color the regions
ax.fill_between([0, 1], [0, 0], [0, 1], alpha=0.05, color="blue",
                label="Dense better")
ax.fill_between([0, 1], [0, 1], [1, 1], alpha=0.05, color="red",
                label="BM25 better")

# ── Plot 2: Histogram of loss differences ───────────────────
ax = axes[1]
ax.hist(loss_diff, bins=40, color="#34495e", alpha=0.7, edgecolor="white")
ax.axvline(0, color="red", linestyle="--", linewidth=2, label="Equal")
ax.axvline(np.mean(loss_diff), color="orange", linestyle="-", linewidth=2,
           label=f"Mean diff = {np.mean(loss_diff):.3f}")
ax.set_xlabel("Loss Difference (BM25 - Dense)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Per-Query Differences\n(positive = Dense is better)")
ax.legend()

# ── Plot 3: Queries sorted by difference ────────────────────
ax = axes[2]
sorted_diff = np.sort(loss_diff)
x = np.arange(len(sorted_diff))
colors_bar = ["#e74c3c" if d > 0 else "#3498db" for d in sorted_diff]
ax.bar(x, sorted_diff, color=colors_bar, width=1.0, alpha=0.7)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Queries (sorted by difference)")
ax.set_ylabel("Loss Difference (BM25 - Dense)")
ax.set_title("Per-Query Advantage\n(Red = Dense better, Blue = BM25 better)")

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["results_dir"], "per_query_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

In [ ]:
THRESHOLD = 0.3  # loss difference threshold

print("=" * 70)
print("  Queries Revealing Inductive Bias Differences")
print("=" * 70)

# Queries where BM25 >> Dense (BM25 loss much lower)
bm25_advantage = [(common_qids[i], loss_diff[i]) for i in range(len(common_qids))
                  if loss_diff[i] < -THRESHOLD]
bm25_advantage.sort(key=lambda x: x[1])

print(f"\n  Queries where BM25 >> Dense ({len(bm25_advantage)} queries):")
print(f"  (Dense's semantic bias FAILS here — exact matching needed)")
for qid, diff in bm25_advantage[:5]:
    query = test_queries[qid]
    bm25_l = evaluations["BM25"]["per_query_losses"][qid]
    dense_l = evaluations["Dense"]["per_query_losses"][qid]
    print(f"    [{qid}] BM25 loss={bm25_l:.3f}, Dense loss={dense_l:.3f}, Δ={diff:.3f}")
    print(f"      Query: \"{query}\"")

# Queries where Dense >> BM25
dense_advantage = [(common_qids[i], loss_diff[i]) for i in range(len(common_qids))
                   if loss_diff[i] > THRESHOLD]
dense_advantage.sort(key=lambda x: -x[1])

print(f"\n  Queries where Dense >> BM25 ({len(dense_advantage)} queries):")
print(f"  (BM25's lexical bias FAILS here — semantic understanding needed)")
for qid, diff in dense_advantage[:5]:
    query = test_queries[qid]
    bm25_l = evaluations["BM25"]["per_query_losses"][qid]
    dense_l = evaluations["Dense"]["per_query_losses"][qid]
    print(f"    [{qid}] BM25 loss={bm25_l:.3f}, Dense loss={dense_l:.3f}, Δ={diff:+.3f}")
    print(f"      Query: \"{query}\"")

print("\n" + "=" * 70)


## Additional experiments for the next submission

This section adds the experiments that the baseline report proposed, plus one stronger stability check that directly matches the research question.

What gets added here:
1. **Split-level retrieval with caching** for train/dev/test.
2. **Hybrid retrieval** with dev-tuned score fusion and an RRF baseline.
3. **Synonym perturbation** to test stability under small query edits.
4. **Vocabulary-gap analysis** to connect lexical matchability to retrieval quality.
5. **Query stratification** by a technicality proxy and by estimated vocabulary gap.
6. **Repeated query-subset evaluation** to measure how much the Dense-vs-BM25 advantage depends on which queries are sampled.
7. **Failure analysis exports** for report tables and qualitative examples.

> Recommended order: run the baseline notebook first through the evaluation section, then run the cells below in order.


In [ ]:
import os
import re
import json
import math
import pickle
import random
from pathlib import Path
from collections import defaultdict, Counter

import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from tqdm.auto import tqdm
from IPython.display import display

# Keep all new experiment artifacts in a separate directory
NEXT_STAGE_CONFIG = {
    "random_seed": 42,
    "top_k_retrieve": TOP_K_RETRIEVE,
    "top_k_overlap": 10,
    "synonym_max_replacements": 1,
    "hybrid_alpha_grid": np.linspace(0.0, 1.0, 11),
    "rrf_k": 60,
    "query_subset_trials": 2000,
    "subset_size": len(test_queries),
}

EXPERIMENT_DIR = os.path.join(CONFIG["results_dir"], "next_stage")
CACHE_DIR = os.path.join(EXPERIMENT_DIR, "cache")
TABLE_DIR = os.path.join(EXPERIMENT_DIR, "tables")
FIG_DIR = os.path.join(EXPERIMENT_DIR, "figures")

for path in [EXPERIMENT_DIR, CACHE_DIR, TABLE_DIR, FIG_DIR]:
    os.makedirs(path, exist_ok=True)

random.seed(NEXT_STAGE_CONFIG["random_seed"])
np.random.seed(NEXT_STAGE_CONFIG["random_seed"])

# Reuse already-computed test retrievals from the baseline section
experiment_results = {
    "BM25": bm25_results,
    "Dense": dense_results,
}
experiment_evaluations = {
    "BM25": evaluations["BM25"],
    "Dense": evaluations["Dense"],
}

# -----------------------------
# Utility helpers
# -----------------------------
def save_pickle(obj, path: str) -> None:
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)


def cache_or_compute(path: str, compute_fn, force: bool = False, verbose: bool = True):
    if os.path.exists(path) and not force:
        if verbose:
            print(f"Loaded cache: {path}")
        return load_pickle(path)

    value = compute_fn()
    save_pickle(value, path)
    if verbose:
        print(f"Saved cache: {path}")
    return value


def ensure_nltk_resource(resource_name: str, lookup_path: str) -> None:
    try:
        nltk.data.find(lookup_path)
    except LookupError:
        nltk.download(resource_name, quiet=True)


def dense_retrieve_batch_chunked(
    queries: dict,
    top_k: int = 100,
    chunk_size: int = 128,
) -> dict:
    """
    Chunked wrapper around dense_retrieve_batch to keep long runs stable and cacheable.
    """
    qids = list(queries.keys())
    results = {}

    for start in tqdm(range(0, len(qids), chunk_size), desc="Dense chunks"):
        batch_qids = qids[start:start + chunk_size]
        batch_queries = {qid: queries[qid] for qid in batch_qids}
        batch_results = dense_retrieve_batch(batch_queries, top_k=top_k)
        results.update(batch_results)

    return results


def sorted_doc_ids(result_dict: dict, top_k: int = 10) -> list:
    return [
        doc_id
        for doc_id, _ in sorted(result_dict.items(), key=lambda x: -x[1])[:top_k]
    ]


def display_metric_table(eval_dict: dict, metric_names: list, title: str = "Metrics"):
    rows = []
    for method_name, eval_obj in eval_dict.items():
        row = {"method": method_name}
        for metric in metric_names:
            row[metric] = eval_obj["aggregate"].get(metric, np.nan)
        rows.append(row)

    df = pd.DataFrame(rows)
    print(f"\n{title}")
    display(df.round(4))
    return df


def get_per_query_metric(eval_obj: dict, metric_name: str = "nDCG@10") -> dict:
    return {qid: metrics[metric_name] for qid, metrics in eval_obj["per_query"].items()}


def export_dataframe(df: pd.DataFrame, filename: str) -> str:
    path = os.path.join(TABLE_DIR, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path


def make_quantile_bins(series: pd.Series, labels: list) -> pd.Series:
    """
    Robust quantile binning even when the raw values contain ties.
    """
    ranked = series.rank(method="first")
    return pd.qcut(ranked, q=len(labels), labels=labels)


def mean_ci(values, alpha: float = 0.05):
    values = np.asarray(values, dtype=float)
    mean = values.mean()
    lo = np.quantile(values, alpha / 2)
    hi = np.quantile(values, 1 - alpha / 2)
    return mean, lo, hi


print("✓ Next-stage experiment helpers are ready.")


In [ ]:

split_queries = {
    "train": train_queries,
    "dev": dev_queries,
    "test": test_queries,
}
split_qrels = {
    "train": train_qrels,
    "dev": dev_qrels,
    "test": test_qrels,
}

split_retrievals = {}
for split_name, query_dict in split_queries.items():
    print(f"\nRetrieving split: {split_name} ({len(query_dict)} queries)")

    if split_name == "test":
        # Reuse the baseline run rather than recomputing it
        bm25_split = experiment_results["BM25"]
        dense_split = experiment_results["Dense"]
    else:
        bm25_cache = os.path.join(CACHE_DIR, f"{split_name}_bm25_results.pkl")
        dense_cache = os.path.join(CACHE_DIR, f"{split_name}_dense_results.pkl")

        bm25_split = cache_or_compute(
            bm25_cache,
            lambda q=query_dict: bm25_retrieve_batch(
                q, top_k=NEXT_STAGE_CONFIG["top_k_retrieve"]
            ),
        )
        dense_split = cache_or_compute(
            dense_cache,
            lambda q=query_dict: dense_retrieve_batch_chunked(
                q,
                top_k=NEXT_STAGE_CONFIG["top_k_retrieve"],
                chunk_size=128,
            ),
        )

    split_retrievals[split_name] = {
        "BM25": bm25_split,
        "Dense": dense_split,
    }

split_evaluations = {}
for split_name in split_queries:
    split_evaluations[split_name] = {}
    for method_name, results_obj in split_retrievals[split_name].items():
        split_evaluations[split_name][method_name] = evaluate_retriever(
            results_obj,
            split_qrels[split_name],
        )

split_rows = []
for split_name, per_method in split_evaluations.items():
    for method_name, eval_obj in per_method.items():
        agg = eval_obj["aggregate"]
        split_rows.append(
            {
                "split": split_name,
                "method": method_name,
                "n_queries": agg["n_queries"],
                "nDCG@10": agg["nDCG@10"],
                "Recall@10": agg["Recall@10"],
                "P@10": agg["P@10"],
                "empirical_risk": agg["empirical_risk"],
                "loss_variance": agg["loss_variance"],
            }
        )

split_metrics_df = pd.DataFrame(split_rows).sort_values(["split", "method"]).reset_index(drop=True)
display(split_metrics_df.round(4))
export_dataframe(split_metrics_df, "split_metrics_baselines.csv")



## Hybrid retrieval

This experiment tests whether BM25 and Dense are complementary rather than purely competing.

- **Tuning rule:** choose the BM25 weight \(\alpha\) on the **dev** split only.
- **Main hybrid:** min-max normalized linear score fusion.
- **Robust baseline:** Reciprocal Rank Fusion (RRF).

This is useful for the report because it shows whether combining two different inductive biases improves average performance and/or reduces variance.


In [ ]:

# ============================================================================
# NEW CELL — Hybrid retrieval: dev-tuned score fusion + RRF baseline
# ============================================================================

def normalize_scores(score_dict: dict, mode: str = "minmax") -> dict:
    """
    Normalize per-query scores before linear fusion.
    """
    if not score_dict:
        return {}

    docs = list(score_dict.keys())
    values = np.array([score_dict[d] for d in docs], dtype=float)

    if mode == "minmax":
        lo, hi = values.min(), values.max()
        if np.isclose(lo, hi):
            norm_values = np.ones_like(values)
        else:
            norm_values = (values - lo) / (hi - lo)
    elif mode == "zscore":
        mean = values.mean()
        std = values.std()
        if np.isclose(std, 0.0):
            norm_values = np.zeros_like(values)
        else:
            norm_values = (values - mean) / std
    else:
        raise ValueError(f"Unknown normalization mode: {mode}")

    return {doc: float(val) for doc, val in zip(docs, norm_values)}


def linear_fuse_results(
    results_a: dict,
    results_b: dict,
    alpha: float = 0.5,
    norm: str = "minmax",
    top_k: int = 100,
) -> dict:
    """
    Query-wise linear score fusion:
        fused = alpha * normalized(BM25) + (1 - alpha) * normalized(Dense)
    """
    fused = {}
    common_qids = sorted(set(results_a) & set(results_b))

    for qid in common_qids:
        a_norm = normalize_scores(results_a[qid], mode=norm)
        b_norm = normalize_scores(results_b[qid], mode=norm)

        all_doc_ids = set(a_norm) | set(b_norm)
        fused_scores = {}
        for doc_id in all_doc_ids:
            fused_scores[doc_id] = alpha * a_norm.get(doc_id, 0.0) + (1.0 - alpha) * b_norm.get(doc_id, 0.0)

        ranked = sorted(fused_scores.items(), key=lambda x: -x[1])[:top_k]
        fused[qid] = {doc_id: float(score) for doc_id, score in ranked}

    return fused


def rrf_fuse_results(
    results_a: dict,
    results_b: dict,
    rrf_k: int = 60,
    top_k: int = 100,
) -> dict:
    """
    Reciprocal Rank Fusion (robust score-free hybrid baseline).
    """
    fused = {}
    common_qids = sorted(set(results_a) & set(results_b))

    for qid in common_qids:
        scores = defaultdict(float)

        for source in [results_a[qid], results_b[qid]]:
            ranked_docs = sorted(source.items(), key=lambda x: -x[1])
            for rank_idx, (doc_id, _) in enumerate(ranked_docs, start=1):
                scores[doc_id] += 1.0 / (rrf_k + rank_idx)

        ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_k]
        fused[qid] = {doc_id: float(score) for doc_id, score in ranked}

    return fused


# ---- Tune alpha on dev only (no test leakage) ----
hybrid_dev_rows = []
for alpha in NEXT_STAGE_CONFIG["hybrid_alpha_grid"]:
    fused_dev = linear_fuse_results(
        split_retrievals["dev"]["BM25"],
        split_retrievals["dev"]["Dense"],
        alpha=float(alpha),
        norm="minmax",
        top_k=NEXT_STAGE_CONFIG["top_k_retrieve"],
    )
    dev_eval = evaluate_retriever(fused_dev, dev_qrels)
    hybrid_dev_rows.append(
        {
            "alpha": float(alpha),
            "nDCG@10": dev_eval["aggregate"]["nDCG@10"],
            "Recall@10": dev_eval["aggregate"]["Recall@10"],
            "P@10": dev_eval["aggregate"]["P@10"],
            "empirical_risk": dev_eval["aggregate"]["empirical_risk"],
        }
    )

hybrid_dev_df = pd.DataFrame(hybrid_dev_rows).sort_values("alpha").reset_index(drop=True)
best_alpha = float(
    hybrid_dev_df.sort_values(
        ["nDCG@10", "Recall@10", "P@10"],
        ascending=False,
    ).iloc[0]["alpha"]
)

print(f"Best dev alpha for linear fusion: {best_alpha:.2f}")
display(hybrid_dev_df.round(4))
export_dataframe(hybrid_dev_df, "hybrid_alpha_sweep_dev.csv")

# ---- Evaluate the selected linear hybrid on all splits ----
for split_name in split_queries:
    fused_split = linear_fuse_results(
        split_retrievals[split_name]["BM25"],
        split_retrievals[split_name]["Dense"],
        alpha=best_alpha,
        norm="minmax",
        top_k=NEXT_STAGE_CONFIG["top_k_retrieve"],
    )
    split_retrievals[split_name]["Hybrid"] = fused_split
    split_evaluations[split_name]["Hybrid"] = evaluate_retriever(
        fused_split,
        split_qrels[split_name],
    )

# Reuse test hybrid in the main experiment dict
experiment_results["Hybrid"] = split_retrievals["test"]["Hybrid"]
experiment_evaluations["Hybrid"] = split_evaluations["test"]["Hybrid"]

# ---- Optional: compare with RRF on test ----
rrf_test_results = rrf_fuse_results(
    experiment_results["BM25"],
    experiment_results["Dense"],
    rrf_k=NEXT_STAGE_CONFIG["rrf_k"],
    top_k=NEXT_STAGE_CONFIG["top_k_retrieve"],
)
rrf_test_eval = evaluate_retriever(rrf_test_results, test_qrels)

# ---- Plot alpha sweep on dev ----
plt.figure(figsize=(7, 4.5))
plt.plot(hybrid_dev_df["alpha"], hybrid_dev_df["nDCG@10"], marker="o", linewidth=2)
plt.axvline(best_alpha, linestyle="--", color="gray", alpha=0.7, label=f"best α={best_alpha:.2f}")
plt.xlabel("BM25 weight α")
plt.ylabel("Dev nDCG@10")
plt.title("Linear Hybrid Tuning on Dev Split")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "hybrid_alpha_sweep_dev.png"), dpi=150, bbox_inches="tight")
plt.show()

# ---- Summary comparison on test ----
hybrid_compare_rows = []
for method_name in ["BM25", "Dense", "Hybrid"]:
    agg = experiment_evaluations[method_name]["aggregate"]
    hybrid_compare_rows.append(
        {
            "method": method_name,
            "nDCG@10": agg["nDCG@10"],
            "Recall@10": agg["Recall@10"],
            "P@10": agg["P@10"],
            "empirical_risk": agg["empirical_risk"],
            "loss_variance": agg["loss_variance"],
        }
    )

hybrid_compare_rows.append(
    {
        "method": f"RRF(k={NEXT_STAGE_CONFIG['rrf_k']})",
        "nDCG@10": rrf_test_eval["aggregate"]["nDCG@10"],
        "Recall@10": rrf_test_eval["aggregate"]["Recall@10"],
        "P@10": rrf_test_eval["aggregate"]["P@10"],
        "empirical_risk": rrf_test_eval["aggregate"]["empirical_risk"],
        "loss_variance": rrf_test_eval["aggregate"]["loss_variance"],
    }
)

hybrid_test_df = pd.DataFrame(hybrid_compare_rows)
display(hybrid_test_df.round(4))
export_dataframe(hybrid_test_df, "hybrid_test_comparison.csv")


# ---- Updated split-level metrics including Hybrid ----
updated_split_rows = []
for split_name, per_method in split_evaluations.items():
    for method_name, eval_obj in per_method.items():
        agg = eval_obj["aggregate"]
        updated_split_rows.append(
            {
                "split": split_name,
                "method": method_name,
                "n_queries": agg["n_queries"],
                "nDCG@10": agg["nDCG@10"],
                "Recall@10": agg["Recall@10"],
                "P@10": agg["P@10"],
                "empirical_risk": agg["empirical_risk"],
                "loss_variance": agg["loss_variance"],
            }
        )

split_metrics_with_hybrid_df = pd.DataFrame(updated_split_rows).sort_values(["split", "method"]).reset_index(drop=True)
display(split_metrics_with_hybrid_df.round(4))
export_dataframe(split_metrics_with_hybrid_df, "split_metrics_with_hybrid.csv")



## Synonym Perturbation (Stability)

> **Note — removed:** This experiment was excluded from the final report due
> to a bug in the synonym-substitution pipeline. The word-replacement function
> did not reliably preserve query semantics while altering BM25 token overlap,
> making the resulting degradation numbers non-credible.
> A trustworthy stability analysis requires manually validated query paraphrases
> or a controlled back-translation protocol — left for future work.


In [ ]:
# ── Synonym perturbation removed (non-credible results due to pipeline bug) ──
#
# The synonym-substitution function could not guarantee semantically equivalent
# replacements, so the degradation figures were unreliable. The variable
# 'synonym_df' is intentionally left undefined; downstream cells that check
# `if "synonym_df" in globals()` will skip synonym-related outputs gracefully.

print("Synonym perturbation section is disabled — see markdown note above.")
print("'synonym_df' is not defined; failure analysis will skip synonym tables.")



## Vocabulary-gap analysis

This section turns the qualitative intuition in the proposal into measurable features.

For each query it computes:
- **Oracle lexical overlap** with relevant documents.
- **Retrieved lexical overlap** for each method's top-10 documents.
- **Technicality proxy** using mean BM25 IDF over query content words.

These features let you test whether Dense gains are concentrated on queries whose relevant documents share little literal vocabulary with the query.


In [ ]:

# ============================================================================
# NEW CELL — Vocabulary-gap analysis
# ============================================================================

def content_tokens(text: str) -> list:
    tokens = [tok.lower() for tok in nltk.word_tokenize(text)]
    return [
        tok
        for tok in tokens
        if TOKEN_PATTERN.match(tok) and tok not in STOPWORDS and len(tok) >= 3
    ]


doc_token_cache = os.path.join(CACHE_DIR, "doc_token_sets.pkl")
doc_token_sets = cache_or_compute(
    doc_token_cache,
    lambda: {
        did: set(content_tokens(
            (corpus[did].get("title", "") + " " + corpus[did].get("text", "")).strip()
        ))
        for did in doc_ids
    },
)

idf_values = list(getattr(bm25, "idf", {}).values())
default_query_idf = float(np.median(idf_values)) if idf_values else 0.0


def query_doc_overlap(query_text: str, doc_id: str) -> float:
    q_tokens = set(content_tokens(query_text))
    if not q_tokens:
        return 0.0
    d_tokens = doc_token_sets.get(doc_id, set())
    return len(q_tokens & d_tokens) / len(q_tokens)


def ranking_mean_overlap(query_text: str, result_dict: dict, top_k: int = 10) -> float:
    ranked_doc_ids = sorted_doc_ids(result_dict, top_k=top_k)
    if not ranked_doc_ids:
        return 0.0
    overlaps = [query_doc_overlap(query_text, did) for did in ranked_doc_ids]
    return float(np.mean(overlaps))


def oracle_relevant_overlap(query_text: str, qrel_dict: dict, reduce: str = "max") -> float:
    relevant_doc_ids = list(qrel_dict.keys())
    overlaps = [query_doc_overlap(query_text, did) for did in relevant_doc_ids]
    if not overlaps:
        return 0.0
    if reduce == "max":
        return float(np.max(overlaps))
    if reduce == "mean":
        return float(np.mean(overlaps))
    raise ValueError("reduce must be 'max' or 'mean'")


def query_technicality_score(query_text: str) -> float:
    q_tokens = content_tokens(query_text)
    if not q_tokens:
        return 0.0
    idfs = [bm25.idf.get(tok, default_query_idf) for tok in q_tokens]
    return float(np.mean(idfs))


vocab_rows = []
for qid, query_text in test_queries.items():
    row = {
        "qid": qid,
        "query": query_text,
        "n_content_tokens": len(set(content_tokens(query_text))),
        "oracle_max_overlap": oracle_relevant_overlap(query_text, test_qrels[qid], reduce="max"),
        "oracle_mean_overlap": oracle_relevant_overlap(query_text, test_qrels[qid], reduce="mean"),
        "technicality_score": query_technicality_score(query_text),
    }
    row["vocab_gap_score"] = 1.0 - row["oracle_max_overlap"]

    for method_name, result_obj in experiment_results.items():
        row[f"{method_name}_retrieved_overlap"] = ranking_mean_overlap(
            query_text,
            result_obj[qid],
            top_k=NEXT_STAGE_CONFIG["top_k_overlap"],
        )
        row[f"{method_name}_nDCG@10"] = experiment_evaluations[method_name]["per_query"][qid]["nDCG@10"]
        row[f"{method_name}_Recall@10"] = experiment_evaluations[method_name]["per_query"][qid]["Recall@10"]

    row["Dense_minus_BM25"] = row["Dense_nDCG@10"] - row["BM25_nDCG@10"]
    if "Hybrid_nDCG@10" in row:
        row["Hybrid_minus_Dense"] = row["Hybrid_nDCG@10"] - row["Dense_nDCG@10"]

    vocab_rows.append(row)

vocab_gap_df = pd.DataFrame(vocab_rows)
export_dataframe(vocab_gap_df, "vocabulary_gap_features.csv")

# ---- Correlation summary ----
corr_rows = []
for method_name in experiment_results.keys():
    rho_overlap, p_overlap = stats.spearmanr(
        vocab_gap_df[f"{method_name}_retrieved_overlap"],
        vocab_gap_df[f"{method_name}_nDCG@10"],
    )
    corr_rows.append(
        {
            "relationship": f"{method_name}: retrieved lexical overlap vs nDCG@10",
            "spearman_rho": rho_overlap,
            "p_value": p_overlap,
        }
    )

rho_gap, p_gap = stats.spearmanr(vocab_gap_df["oracle_max_overlap"], vocab_gap_df["Dense_minus_BM25"])
corr_rows.append(
    {
        "relationship": "oracle lexical overlap vs (Dense - BM25) nDCG@10",
        "spearman_rho": rho_gap,
        "p_value": p_gap,
    }
)

corr_df = pd.DataFrame(corr_rows)
display(corr_df.round(4))
export_dataframe(corr_df, "vocabulary_gap_correlations.csv")

# ---- Plot overlap-performance relationships ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) Does Dense gain more when oracle lexical overlap is low?
axes[0].scatter(
    vocab_gap_df["oracle_max_overlap"],
    vocab_gap_df["Dense_minus_BM25"],
    alpha=0.7,
    edgecolors="white",
)
axes[0].axhline(0.0, linestyle="--", color="gray", alpha=0.7)
axes[0].set_xlabel("Max lexical overlap with relevant documents")
axes[0].set_ylabel("Dense - BM25 (nDCG@10)")
axes[0].set_title("Performance Gap vs. Inherent Vocabulary Matchability")
axes[0].grid(alpha=0.3)

# (2) Retrieved overlap vs own performance
axes[1].scatter(
    vocab_gap_df["BM25_retrieved_overlap"],
    vocab_gap_df["BM25_nDCG@10"],
    alpha=0.65,
    label="BM25",
    color=COLORS.get("BM25", "gray"),
    edgecolors="white",
)
axes[1].scatter(
    vocab_gap_df["Dense_retrieved_overlap"],
    vocab_gap_df["Dense_nDCG@10"],
    alpha=0.65,
    label="Dense",
    color=COLORS.get("Dense", "gray"),
    edgecolors="white",
)

if "Hybrid" in experiment_results:
    axes[1].scatter(
        vocab_gap_df["Hybrid_retrieved_overlap"],
        vocab_gap_df["Hybrid_nDCG@10"],
        alpha=0.65,
        label="Hybrid",
        color=COLORS.get("Hybrid", "gray"),
        edgecolors="white",
    )

axes[1].set_xlabel("Mean lexical overlap of top-10 retrieved documents")
axes[1].set_ylabel("nDCG@10")
axes[1].set_title("How Lexical Matchability Relates to Each Method's Quality")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "vocabulary_gap_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================================
# NEW CELL — Query stratification by technicality and vocabulary gap
# ============================================================================

technicality_labels = [
    "Low IDF / plainer",
    "Mid IDF",
    "High IDF / technical",
]
gap_labels = [
    "Low gap",
    "Medium gap",
    "High gap",
]

vocab_gap_df["technicality_bin"] = make_quantile_bins(
    vocab_gap_df["technicality_score"],
    technicality_labels,
)
vocab_gap_df["gap_bin"] = make_quantile_bins(
    vocab_gap_df["vocab_gap_score"],
    gap_labels,
)

methods_for_strata = list(experiment_results.keys())


def stratified_metric_table(
    df: pd.DataFrame,
    stratum_col: str,
    methods: list,
    metric_name: str = "nDCG@10",
) -> pd.DataFrame:
    rows = []
    for stratum_value, group in df.groupby(stratum_col, observed=False):
        row = {
            "stratum": str(stratum_value),
            "n_queries": len(group),
        }
        for method_name in methods:
            row[f"{method_name}_{metric_name}"] = group[f"{method_name}_{metric_name}"].mean()
        if "BM25" in methods and "Dense" in methods:
            row["Dense_minus_BM25"] = row[f"Dense_{metric_name}"] - row[f"BM25_{metric_name}"]
        if "Hybrid" in methods and "Dense" in methods:
            row["Hybrid_minus_Dense"] = row[f"Hybrid_{metric_name}"] - row[f"Dense_{metric_name}"]
        rows.append(row)

    return pd.DataFrame(rows)


technicality_df = stratified_metric_table(
    vocab_gap_df,
    "technicality_bin",
    methods_for_strata,
    metric_name="nDCG@10",
)
gap_df = stratified_metric_table(
    vocab_gap_df,
    "gap_bin",
    methods_for_strata,
    metric_name="nDCG@10",
)

display(technicality_df.round(4))
display(gap_df.round(4))

export_dataframe(technicality_df, "technicality_stratification_ndcg10.csv")
export_dataframe(gap_df, "vocabulary_gap_stratification_ndcg10.csv")

# ---- Plot stratified comparisons ----
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Technicality bins
x = np.arange(len(technicality_df))
width = 0.22
for idx, method_name in enumerate(methods_for_strata):
    axes[0].bar(
        x + idx * width,
        technicality_df[f"{method_name}_nDCG@10"],
        width,
        label=method_name,
        color=COLORS.get(method_name, "gray"),
        alpha=0.85,
        edgecolor="white",
    )

axes[0].set_xticks(x + width * (len(methods_for_strata) - 1) / 2)
axes[0].set_xticklabels(technicality_df["stratum"], rotation=15)
axes[0].set_ylabel("Mean nDCG@10")
axes[0].set_title("Performance by Query Technicality Proxy")
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend()

# Vocabulary-gap bins
x = np.arange(len(gap_df))
for idx, method_name in enumerate(methods_for_strata):
    axes[1].bar(
        x + idx * width,
        gap_df[f"{method_name}_nDCG@10"],
        width,
        label=method_name,
        color=COLORS.get(method_name, "gray"),
        alpha=0.85,
        edgecolor="white",
    )

axes[1].set_xticks(x + width * (len(methods_for_strata) - 1) / 2)
axes[1].set_xticklabels(gap_df["stratum"])
axes[1].set_ylabel("Mean nDCG@10")
axes[1].set_title("Performance by Estimated Vocabulary Gap")
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "query_stratification.png"), dpi=150, bbox_inches="tight")
plt.show()



## Query-sample sensitivity

This is the strongest direct test of the research question.

Instead of relying on one test split, the code merges train/dev/test queries, then repeatedly samples random subsets of the same size as the official test set. For each subset it recomputes mean **nDCG@10** for BM25, Dense, and Hybrid.

This gives a distribution over:
- subset-level performance for each method,
- **Dense - BM25** performance gaps,
- optionally **Hybrid - Dense** gaps.

That distribution tells you whether the observed advantage is stable or heavily sample-dependent.


In [ ]:

# ============================================================================
# NEW CELL — Query-sample sensitivity via repeated subset evaluation
# ============================================================================

def prefix_split_ids(split_name: str, queries: dict, qrels: dict):
    prefixed_queries = {}
    prefixed_qrels = {}
    for qid, query_text in queries.items():
        new_qid = f"{split_name}::{qid}"
        prefixed_queries[new_qid] = query_text
        prefixed_qrels[new_qid] = qrels[qid]
    return prefixed_queries, prefixed_qrels


pref_train_queries, pref_train_qrels = prefix_split_ids("train", train_queries, train_qrels)
pref_dev_queries, pref_dev_qrels = prefix_split_ids("dev", dev_queries, dev_qrels)
pref_test_queries, pref_test_qrels = prefix_split_ids("test", test_queries, test_qrels)

all_queries_prefixed = {}
all_queries_prefixed.update(pref_train_queries)
all_queries_prefixed.update(pref_dev_queries)
all_queries_prefixed.update(pref_test_queries)

all_qrels_prefixed = {}
all_qrels_prefixed.update(pref_train_qrels)
all_qrels_prefixed.update(pref_dev_qrels)
all_qrels_prefixed.update(pref_test_qrels)

print(f"Total queries across all splits: {len(all_queries_prefixed)}")

all_bm25_results = cache_or_compute(
    os.path.join(CACHE_DIR, "all_queries_bm25_results.pkl"),
    lambda: bm25_retrieve_batch(
        all_queries_prefixed,
        top_k=NEXT_STAGE_CONFIG["top_k_retrieve"],
    ),
)
all_dense_results = cache_or_compute(
    os.path.join(CACHE_DIR, "all_queries_dense_results.pkl"),
    lambda: dense_retrieve_batch_chunked(
        all_queries_prefixed,
        top_k=NEXT_STAGE_CONFIG["top_k_retrieve"],
        chunk_size=128,
    ),
)

all_results = {
    "BM25": all_bm25_results,
    "Dense": all_dense_results,
}
if "Hybrid" in experiment_results:
    all_results["Hybrid"] = linear_fuse_results(
        all_bm25_results,
        all_dense_results,
        alpha=best_alpha,
        norm="minmax",
        top_k=NEXT_STAGE_CONFIG["top_k_retrieve"],
    )

all_evaluations = {
    method_name: evaluate_retriever(result_obj, all_qrels_prefixed)
    for method_name, result_obj in all_results.items()
}

resample_methods = list(all_results.keys())
common_all_qids = sorted(
    set.intersection(*[set(all_evaluations[m]["per_query"].keys()) for m in resample_methods])
)

metric_arrays = {
    method_name: np.array(
        [all_evaluations[method_name]["per_query"][qid]["nDCG@10"] for qid in common_all_qids],
        dtype=float,
    )
    for method_name in resample_methods
}


def repeated_subset_evaluation(
    metric_arrays: dict,
    subset_size: int,
    n_trials: int = 2000,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n_total = len(next(iter(metric_arrays.values())))
    rows = []

    for trial in tqdm(range(n_trials), desc="Subset resampling"):
        indices = rng.choice(n_total, size=subset_size, replace=False)
        row = {"trial": trial}
        for method_name, values in metric_arrays.items():
            row[method_name] = float(np.mean(values[indices]))

        if "BM25" in metric_arrays and "Dense" in metric_arrays:
            row["Dense_minus_BM25"] = row["Dense"] - row["BM25"]
        if "Hybrid" in metric_arrays and "Dense" in metric_arrays:
            row["Hybrid_minus_Dense"] = row["Hybrid"] - row["Dense"]

        rows.append(row)

    return pd.DataFrame(rows)


subset_df = repeated_subset_evaluation(
    metric_arrays,
    subset_size=NEXT_STAGE_CONFIG["subset_size"],
    n_trials=NEXT_STAGE_CONFIG["query_subset_trials"],
    seed=NEXT_STAGE_CONFIG["random_seed"],
)

export_dataframe(subset_df, "query_subset_resampling.csv")

summary_rows = []
for method_name in resample_methods:
    mean_val, lo, hi = mean_ci(subset_df[method_name].to_numpy())
    summary_rows.append(
        {
            "method": method_name,
            "subset_size": NEXT_STAGE_CONFIG["subset_size"],
            "n_trials": NEXT_STAGE_CONFIG["query_subset_trials"],
            "mean_subset_nDCG@10": mean_val,
            "q025": lo,
            "q975": hi,
            "std_subset_nDCG@10": subset_df[method_name].std(ddof=1),
        }
    )

if "Dense_minus_BM25" in subset_df:
    summary_rows.append(
        {
            "method": "Dense_minus_BM25",
            "subset_size": NEXT_STAGE_CONFIG["subset_size"],
            "n_trials": NEXT_STAGE_CONFIG["query_subset_trials"],
            "mean_subset_nDCG@10": subset_df["Dense_minus_BM25"].mean(),
            "q025": subset_df["Dense_minus_BM25"].quantile(0.025),
            "q975": subset_df["Dense_minus_BM25"].quantile(0.975),
            "std_subset_nDCG@10": subset_df["Dense_minus_BM25"].std(ddof=1),
        }
    )

if "Hybrid_minus_Dense" in subset_df:
    summary_rows.append(
        {
            "method": "Hybrid_minus_Dense",
            "subset_size": NEXT_STAGE_CONFIG["subset_size"],
            "n_trials": NEXT_STAGE_CONFIG["query_subset_trials"],
            "mean_subset_nDCG@10": subset_df["Hybrid_minus_Dense"].mean(),
            "q025": subset_df["Hybrid_minus_Dense"].quantile(0.025),
            "q975": subset_df["Hybrid_minus_Dense"].quantile(0.975),
            "std_subset_nDCG@10": subset_df["Hybrid_minus_Dense"].std(ddof=1),
        }
    )

subset_summary_df = pd.DataFrame(summary_rows)
display(subset_summary_df.round(4))
export_dataframe(subset_summary_df, "query_subset_resampling_summary.csv")

# ---- Plot subset performance distributions ----
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

for method_name in resample_methods:
    axes[0].hist(
        subset_df[method_name],
        bins=35,
        alpha=0.45,
        label=method_name,
        color=COLORS.get(method_name, "gray"),
        edgecolor="white",
        density=True,
    )
axes[0].set_xlabel("Subset mean nDCG@10")
axes[0].set_ylabel("Density")
axes[0].set_title(f"Distribution across random query subsets (n={NEXT_STAGE_CONFIG['subset_size']})")
axes[0].legend()

if "Dense_minus_BM25" in subset_df:
    observed_test_delta = (
        experiment_evaluations["Dense"]["aggregate"]["nDCG@10"]
        - experiment_evaluations["BM25"]["aggregate"]["nDCG@10"]
    )
    axes[1].hist(
        subset_df["Dense_minus_BM25"],
        bins=35,
        alpha=0.8,
        edgecolor="white",
    )
    axes[1].axvline(0.0, linestyle="--", color="gray", alpha=0.7, label="tie")
    axes[1].axvline(
        observed_test_delta,
        linestyle="-",
        color="black",
        alpha=0.9,
        label=f"test Δ={observed_test_delta:.4f}",
    )
    axes[1].set_xlabel("Dense - BM25 subset mean nDCG@10")
    axes[1].set_ylabel("Count")
    axes[1].set_title("How sensitive is the advantage to query sampling?")
    axes[1].legend()
else:
    axes[1].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "query_subset_resampling.png"), dpi=150, bbox_inches="tight")
plt.show()

if "Dense_minus_BM25" in subset_df:
    prob_dense_beats = float(np.mean(subset_df["Dense_minus_BM25"] > 0))
    print(f"P(Dense > BM25) over random query subsets: {prob_dense_beats:.3f}")

if "Hybrid_minus_Dense" in subset_df:
    prob_hybrid_beats = float(np.mean(subset_df["Hybrid_minus_Dense"] > 0))
    print(f"P(Hybrid > Dense) over random query subsets: {prob_hybrid_beats:.3f}")



## Failure Analysis

This section exports concrete queries where:
- Dense strongly outperforms BM25 (semantic gap favours bi-encoder),
- BM25 strongly outperforms Dense (exact match favours sparse retrieval),
- Hybrid improves over both baselines (complementarity in action).

These tables provide qualitative evidence to accompany the quantitative analysis.


In [ ]:

# ============================================================================
# NEW CELL — Failure analysis tables and qualitative examples
# ============================================================================

test_analysis_df = vocab_gap_df.copy()
test_analysis_df["BM25_loss"] = 1.0 - test_analysis_df["BM25_nDCG@10"]
test_analysis_df["Dense_loss"] = 1.0 - test_analysis_df["Dense_nDCG@10"]

if "Hybrid_nDCG@10" in test_analysis_df.columns:
    test_analysis_df["Hybrid_loss"] = 1.0 - test_analysis_df["Hybrid_nDCG@10"]
    test_analysis_df["Hybrid_rescue_over_best_base"] = (
        test_analysis_df["Hybrid_nDCG@10"]
        - test_analysis_df[["BM25_nDCG@10", "Dense_nDCG@10"]].max(axis=1)
    )


def top_ranked_with_relevance(results_dict: dict, qid: str, qrels: dict, top_k: int = 5) -> list:
    ranked = sorted(results_dict[qid].items(), key=lambda x: -x[1])[:top_k]
    rows = []
    for rank, (doc_id, score) in enumerate(ranked, start=1):
        rows.append(
            {
                "rank": rank,
                "doc_id": doc_id,
                "score": float(score),
                "relevance": qrels[qid].get(doc_id, 0),
                "title": corpus[doc_id].get("title", "")[:120],
            }
        )
    return rows


def show_case_details(qid: str):
    print("=" * 100)
    print(f"QID: {qid}")
    print(f"Query: {test_queries[qid]}")
    print(f"Relevant docs: {len(test_qrels[qid])}")

    for method_name, result_obj in experiment_results.items():
        print(f"\n{method_name} top-5:")
        case_df = pd.DataFrame(top_ranked_with_relevance(result_obj, qid, test_qrels, top_k=5))
        display(case_df.round(4))


# Biggest Dense wins
dense_advantage_cases = (
    test_analysis_df.sort_values("Dense_minus_BM25", ascending=False)
    .loc[:, ["qid", "query", "oracle_max_overlap", "technicality_score", "BM25_nDCG@10", "Dense_nDCG@10", "Dense_minus_BM25"]]
    .head(15)
)
export_dataframe(dense_advantage_cases, "failure_cases_dense_advantage.csv")

# Biggest BM25 wins
bm25_advantage_cases = (
    test_analysis_df.sort_values("Dense_minus_BM25", ascending=True)
    .loc[:, ["qid", "query", "oracle_max_overlap", "technicality_score", "BM25_nDCG@10", "Dense_nDCG@10", "Dense_minus_BM25"]]
    .head(15)
)
export_dataframe(bm25_advantage_cases, "failure_cases_bm25_advantage.csv")

# Queries where hybrid helps beyond both base methods
if "Hybrid_rescue_over_best_base" in test_analysis_df.columns:
    hybrid_rescue_cases = (
        test_analysis_df.sort_values("Hybrid_rescue_over_best_base", ascending=False)
        .loc[:, ["qid", "query", "BM25_nDCG@10", "Dense_nDCG@10", "Hybrid_nDCG@10", "Hybrid_rescue_over_best_base"]]
        .head(15)
    )
    export_dataframe(hybrid_rescue_cases, "failure_cases_hybrid_rescue.csv")
    display(hybrid_rescue_cases.round(4))

# Queries that are especially brittle under synonym perturbation
if "synonym_df" in globals():
    bm25_brittle = synonym_df.sort_values("BM25_drop", ascending=False).head(15)
    export_dataframe(bm25_brittle, "failure_cases_synonym_brittle_bm25.csv")

print("Top Dense-advantage queries:")
display(dense_advantage_cases.round(4))

print("Top BM25-advantage queries:")
display(bm25_advantage_cases.round(4))

# Show a few detailed examples inline
for qid in dense_advantage_cases["qid"].head(2):
    show_case_details(qid)

for qid in bm25_advantage_cases["qid"].head(2):
    show_case_details(qid)
